# RSNA Knee Stack3 — nartaa 0.949 + ConvNeXt 2.5D + knee_meta v4 (per-label rank stack)

Stage A (nartaa cell, verbatim) writes `submission.csv` + `_sota_0949.csv` first.
Any later stage failure falls back to that 0.949 base (atomic swap only on success).
Stack weights: shrunk per-label simplex fit on the 58-gold gate (shrunk 50% toward
uniform). Override with env `RSNA_STACK_MODE` = shrunk | uniform3 | uniform2.

# RSNA Knee: accuracy inference (0.949 public AUC)

**Runnable inference notebook.** The exact checkpoint is attached and a T4 GPU is enabled. This version runs the scored pipeline and writes `submission.csv`.

Original 384-pixel full-field training; native 320-pixel center crop at inference, K94, two views with equal plain/anatomical-mirror probability averaging. One seed-42 training run, one averaged checkpoint at inference; averaged epochs [14, 11, 12]. The accuracy model is the original 384-trained checkpoint: the later native-320-crop-trained alternative scored 0.946 and is not substituted here.

The score is the displayed ordinary full-12 public leaderboard macro-AUC for submission `56906959`. The observed 26.1 minutes is submit-to-scored turnaround including queueing, orchestration and polling, not official worker runtime or an efficiency rank. Gold58 was reused for selection and development; it is not independent CV. This release does not establish significance or private-test performance.

The scored source's opening docstring and some comments describe older Raptor versions (0.924, 64 slices, 42 windows and earlier ensembles). They are retained to preserve the complete scored source bytes. This introduction, the live constants and the manifest describe the selected recipe: 96 slices, 94 windows, and one checkpoint.

## Reproduction requirements

Accept the competition rules and attach `rsna-knee-abnormality-detection`. Attach the exact checkpoint `raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt` from the [selected SWA weights](https://www.kaggle.com/datasets/nartaa/rsna-knee-publication-swa-weights-20261007), version 1. This public dependency contains only learned checkpoints and aggregate metadata. OAI images, OAI caches, report text and training label tables are not inference inputs and are not distributed.

Use the original Kaggle GPU environment: two Tesla T4 GPUs were available in the scored machine; the unchanged source uses its original CUDA worker policy. Runtime dependencies are Python, PyTorch with CUDA, timm, NumPy, pandas, pydicom and OpenCV. No downloads or pip installs are added. The scored Kaggle container is pinned to `gcr.io/kaggle-private-byod/python@sha256:2757e0c7d1e0a9cb43da657b97e223c321a98f5014bdf64f44f2f6b083ad2b2f`. The associated Kaggle qualification recorded torch 2.11.0+cu128 and timm 1.0.29.

Source SHA-256: `bcd864cbf349882e876959b49cb348cbb89b4c62fc204d4e4261049c5e3a79c5`. Checkpoint SHA-256: `7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef`. The original private scored source is `nartaa/rsna-knee-original384-s42-mirror320-infer`, version 1. Its private dependency is `nartaa/rsna-knee-oai-original384-s42-pair`, version 1; it remains private. This publication uses a separate notebook slug and leaves the private scored notebook unchanged.

The metadata enables T4 GPUs with internet disabled. `RUN_INFERENCE = True` runs the two code cells in order after verifying the attached checkpoint. The final source writes `submission.csv`; competition submission is managed separately.

## Credits and required acknowledgement

Built on dreaddevelopment's [Raptor CoAtNet/MIL inference recipe and public baseline weights](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense). These are our retrained checkpoints, rather than those baseline weights. Weak-label sources include [stevenleehans](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels), [pilkwang](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels), and [riadmohamed42's JEV labels and verification](https://www.kaggle.com/code/riadmohamed42/jev-knee-labels-verification), combined with our report-derived labels. Source reports and study-level label tables are not included here. The CoAtNet backbone comes from timm. Competition data is provided under the [RSNA Knee competition rules](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules).

Data and/or research tools used in the preparation of this manuscript were obtained and analyzed from the controlled access datasets distributed from the Osteoarthritis Initiative (OAI), a data repository housed within the NIMH Data Archive (NDA). OAI is a collaborative informatics system created by the National Institute of Mental Health and the National Institute of Arthritis, Musculoskeletal and Skin Diseases (NIAMS) to provide a worldwide resource to quicken the pace of biomarker identification, scientific investigation and OA drug development. Dataset identifier(s): 10.15154/0hcg-f676.

[Shared NDA Study](https://nda.nih.gov/study.html?id=3407) · [DOI: 10.15154/0hcg-f676](https://doi.org/10.15154/0hcg-f676). Official acknowledgement source: [NDA manuscript preparation](https://nda.nih.gov/nda/manuscript-preparation).

In [ ]:
# Authorized scoring run: verify the selected checkpoint before inference.
RUN_INFERENCE = True
if not RUN_INFERENCE:
    raise SystemExit("Static notebook only. Attach the exact checkpoint and competition, enable a T4 GPU, and set RUN_INFERENCE = True to run inference.")

import hashlib
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("This scored pipeline requires CUDA; CPU inference is not enabled.")
_weight_name = 'raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt'
_candidates = [
    Path('/kaggle/input/rsna-knee-publication-swa-weights-20261007') / _weight_name,
    Path('/kaggle/input/datasets/nartaa/rsna-knee-publication-swa-weights-20261007') / _weight_name,
    Path('/kaggle/input/rsna-knee-oai-original384-s42-pair') / _weight_name,
    Path('/kaggle/input/datasets/nartaa/rsna-knee-oai-original384-s42-pair') / _weight_name,
]
_weights = next((p for p in _candidates if p.is_file()), None)
if _weights is None:
    raise FileNotFoundError("Attach the exact selected weights; no alternative model is substituted.")

with _weights.open('rb') as _f:
    if hasattr(hashlib, "file_digest"):
        digest = hashlib.file_digest(_f, 'sha256').hexdigest()
    else:
        h = hashlib.sha256()
        while chunk := _f.read(65536):
            h.update(chunk)
        digest = h.hexdigest()
    assert digest == '7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef', f"Checkpoint hash differs: got {digest}"
print(f"Verified verified SOTA weights: {_weights} (SHA-256 {digest})")

In [ ]:
# %% [code]
#!/usr/bin/env python3
"""Knee MRI: twelve findings from a single model

This notebook takes a knee MRI study and scores twelve findings at once: ACL tear, MCL tear,
medial and lateral meniscus tears, osteoarthritis in the medial, lateral and patellofemoral
compartments, joint effusion, synovitis, a Baker's cyst, bone contusion and fracture. It scores
0.924 on the public leaderboard using one model, with no ensembling and no test-time augmentation.

This is the inference half of the work. The model was trained separately and its weights are
attached as a dataset, so this notebook only loads them and predicts:
https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense

Where the training labels came from

Worth saying up front, because it shapes everything else. The competition gives you 4,407 studies
but structured labels for only 58 of them. Every other study arrives with a free-text radiology
report and nothing more, so there is very little to train against out of the box.

The labels behind these weights were made by reading those reports with a language model and
turning each into twelve probabilities rather than twelve yes or no answers. A report that says a
tear is suspected becomes a number near 0.8, not a 1, which is a fairer target than forcing every
hedged sentence into a hard label. That yields 4,349 studies to train on. The 58 studies that came
with real labels were never trained on and are used to check the result honestly; the model reaches
0.9167 macro-AUC on them.

Building a fixed input from studies that are all shaped differently

The hard part of this competition is not the network, it is that no two studies look alike. A
study holds several DICOM series shot in different planes, the number of series varies, and the
number of slices in a series varies more. Anything that expects a fixed-size input has to be given
one.

The approach here is to fill five fixed slots per study, always in the same order, for a stack of
64 images:

  18 slices from a sagittal series, preferring a fluid-sensitive one
  14 slices from a second sagittal series, preferring one that is not fluid-sensitive
  12 slices from a coronal series, preferring a fluid-sensitive one
   8 slices from a second coronal series
  12 slices from an axial series

Preferring a fluid-sensitive series for some slots and not for others is deliberate. Fluid-
sensitive sequences show swelling, effusion and acute injury clearly, while the other sequences
show anatomy and cartilage better, and the twelve findings are split across both. If a study has
no series for a slot, the slot is left as zeros and the model is told to skip it rather than being
fed something misleading.

Within a series, slices are taken evenly across 6 to 94 percent of the stack rather than from the
middle. The outer slices are where the collateral ligaments and the lateral meniscus sit, and
cutting them was measurably costing accuracy on exactly those findings.

Every slice is cropped to a 140 mm box around the centre of the image using the pixel spacing from
the DICOM header, then resized to 336 pixels. Cropping by millimetres rather than by pixel count
matters: it means a knee occupies the same fraction of the frame whether the scan was acquired at
0.3 or 0.5 mm per pixel, so the model is not asked to learn scale differences that carry no medical
information.

How the model reads the stack

Three neighbouring slices are stacked into the three channels of one image. The network then sees
a little of what lies above and below the slice in the middle, which is most of the benefit of a 3D
model at the cost of a 2D one. Each of these three-slice windows is passed through a CoAtNet
backbone at 384 pixels.

The windows are combined with an attention layer that has separate weights for each of the twelve
findings. This is the part that matters most. A cruciate tear may be visible on two sagittal slices
while osteoarthritis is spread across many coronal ones, and a single pooled score forces those two
to share one notion of which slices are important. Giving each finding its own attention weights
lets each one draw on the slices that actually show it.

Running it

Scoring uses 42 windows per study. Inference runs in half precision and automatically retries a
study in full precision if it fails, so no study is ever dropped from the submission. The notebook
needs no internet: the backbone is loaded from the attached weights rather than downloaded.
"""
import os, sys, glob, time, json, gc
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import threading, queue
import torch, torch.nn as nn, torch.nn.functional as F
import timm
# T4 (Turing) cuDNN v9 has fp16/fp32 conv engines but NOT bf16 for these shapes
# ("GET was unable to find an engine..."); benchmark lets it pick a valid algo for
# the fixed (1,24,3,res,res) input.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# ---- fixed config (must match training exactly) -----------------------------
INPUT_RES = 320
INPUT_MODE = "crop"
IMG = 384   # native 384 corpus: the model saw 384 px crops directly, no 336->384 upsample
CROP_MM = 140.0
# 96 slices per study at native 384 px. Must match the corpus the weights were trained on
# (cache/build_corpus96.py).
SLOTS = [("Sagittal", 1, 26), ("Sagittal", 0, 22), ("Coronal", 1, 18),
         ("Coronal", 0, 12), ("Axial", -1, 18)]
MAXS = sum(s[2] for s in SLOTS)                     # 96
K_EVAL = 94   # every window position of the 96-slice test volume (full coverage)
HDR_THREADS = 16   # per prep process: header reads are fuse-latency-bound, not CPU-bound
PX_THREADS = 8     # per prep process: pixel reads of the ~96 picked slices
QUEUE_MAX = 48     # studies in flight between the prep pool and the GPU consumers (RAM bound: ~14 MB each)
NORM = "imagenet"
FASTREAD_VARIANT = 'study_lut'   # kernel v3: research/fastread/readers.py, byte-identical to build_study
FR_HDR, FR_PX = 32, 16
_FASTREAD_B64 = "..."
def _load_fastread():
    import base64 as _b, importlib.util as _iu
    p = '/kaggle/working/_fastread.py' if os.path.isdir('/kaggle/working') else os.path.join(os.path.dirname(os.path.abspath(__file__)), '_fastread.py')
    if not os.path.exists(p):
        open(p, 'wb').write(_b.b64decode(_FASTREAD_B64))
    spec = _iu.spec_from_file_location('_fastread', p); m = _iu.module_from_spec(spec); spec.loader.exec_module(m)
    return m
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Three arms: (weights filename, fallback arch, fallback res). ck carries arch+res too.
# Selected 2026-08-19 by greedy forward selection AND exhaustive subset search over a 7-arm
# panel on the 45-study gold set (phase2/blend_panel.py); both agree on this exact set.
# Singles: coatnet384 0.9025 | swinbase384 0.8825 | effv2l480 0.8716.
# Blend {coatnet+swin+effv2l} = 0.9068 (2-arm {coatnet+swin} = 0.9059, coatnet alone 0.9025).
# Dropped as redundant: cnn336 (0.8833, the former champion), cnbase384 (0.8754),
# cnlarge384 (0.8752), maxvit384 (0.8438).
#
# SINGLE ARM: coatnet_rmlp_2_rw_384 retrained on the EXPANDED 4,349-study corpus.
#
# Why one arm and not the 3-arm blend: on the live leaderboard CoAtNet alone scored 0.914 while
# every blend scored 0.914-0.915, so ensembling is worth ~+0.001 there -- the ~+0.010 it showed
# on the old 45-study gold set was gold-set noise. One arm is also 1/3 the kernel runtime.
#
# Corpus expansion: the corpus previously held 3,200 of the 4,349 labelled studies and only 45
# of the 58 gold studies. Rebuilt to 4,407 studies (+37.8% training data, 58-study gate).
#
# Measured on the 58-study gate (the incumbent re-scored on the SAME gate for a fair compare):
#   incumbent CoAtNet (3,155-study corpus) 0.8923
#   this model       (4,349-study corpus) 0.9054   (+0.0131, better in 92.7% of 2000 bootstraps)
# Biggest gains land on the findings that were capping us: Lateral Meniscus +0.071,
# Fracture +0.057, Lateral OA +0.048, Medial Meniscus +0.035, ACL +0.028.
ARMS = [   # our 5-fold CoAtNet (public recipe reproduced on our folds/labels), equal-weight rank-mean
    {"file": "raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt", "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384, "w": 1.0},
]


# ============================================================================
# Model -- verbatim from finetune_raptor.py
# ============================================================================
def build_backbone(arch, pretrained=False):
    # maxvit/maxxvit/coatnet are conv-attention hybrids: NO CLS token, NO interpolatable
    # pos-embed -> avg pool. The "vit" substring in "coatnet"/"maxvit" must NOT route them
    # down the ViT path (mirrors finetune_raptor.py exactly).
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3, img_size=INPUT_RES)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)


class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        # Bound backbone memory; the attention head still sees every window together.
        flat = x.flatten(0, 1)
        f = torch.cat([self.backbone(chunk) for chunk in flat.split(16)], dim=0)
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))


def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    import hashlib
    with open(pt_path, "rb") as stream:
        observed_sha = hashlib.file_digest(stream, "sha256").hexdigest()
    assert observed_sha == '7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef', "Bound checkpoint bytes differ"
    print(f"[weights] sha256 {observed_sha}", flush=True)
    ck = torch.load(pt_path, map_location="cpu", weights_only=False)
    arch = ck.get("arch", arch_default)
    assert int(ck.get("res", res_default)) == 384, "Expected original 384 checkpoint"
    ck_res = INPUT_RES
    bb = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(device)
    # NOTE: DataParallel removed on purpose. On the full hidden test it drove a system-RAM OOM
    # (per-forward module replication over many studies); a single T4 handles K_EVAL=24 windows
    # fine. Arms are also run SEQUENTIALLY (see main) so peak RAM == one model, not two.
    del ck
    gc.collect()
    return model, ck_res


# ============================================================================
# Eval windowing -- verbatim from finetune_raptor.py StudyWindows (train=False)
# ============================================================================
def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]


SLOT_SIZES = [26, 22, 18, 12, 18]
SAGITTAL_SLOTS = (0, 1)

def _slot_index(D):
    b = np.cumsum([0] + SLOT_SIZES)
    if b[-1] != D: raise RuntimeError(f'slot sizes sum to {b[-1]}, volume has {D} slices')
    s_ = np.searchsorted(b, np.arange(D), side='right') - 1
    rel = (np.arange(D) - b[s_]) / np.maximum(1, np.array(SLOT_SIZES)[s_] - 1)
    return s_, rel

def _mirror_tri(tri, cs, D):
    """final-g1-bundle MIRROR_TTA: true L/R mirror of (k,3,H,W) uint8 eval triplets (research/rnd4/mirror/mirror_f0.py). The slot of each
    window's CENTRE slice decides the axis: sagittal (slots 0,1) -> reverse the 3 slice-channels; coronal/axial (2,3,4) -> flip the width.
    Done on uint8 BEFORE the per-channel LUT, i.e. 'mirror the knee, then normalise'."""
    s_, _ = _slot_index(D)
    sag = torch.from_numpy(np.isin(s_[np.asarray(cs, dtype=np.int64)], SAGITTAL_SLOTS)).to(tri.device).view(-1, 1, 1, 1)
    return torch.where(sag, tri.flip(1), tri.flip(-1))

def infer_pair_probs(model, plain, mirrored, device):
    assert plain.shape==mirrored.shape==(94,3,320,320)
    assert all(not m.training for m in model.modules()), "TTA must not update BN or enable dropout"
    p = infer_probs(model, plain, device)
    q = infer_probs(model, mirrored, device)
    return 0.5 * (p + q)

def eval_windows(vol, mask, k, res, norm=NORM, mirror=False):
    D = vol.shape[0]
    cs = _eval_centers(mask, D, k)
    wins = np.empty((len(cs), 3, res, res), np.float32)
    for j, c in enumerate(cs):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0)
        if mirror:
            tri = _mirror_tri(torch.from_numpy(tri)[None], [c], D)[0].numpy()
        tri = tri.astype(np.float32) / 255.0
        assert tri.shape[-2:] == (IMG, IMG) and res == INPUT_RES
        lo = (IMG - res) // 2
        tri = tri[:, lo:lo + res, lo:lo + res]
        t = torch.from_numpy(np.ascontiguousarray(tri))
        if t.shape[-1] != res:
            t = F.interpolate(t[None], size=(res, res), mode="bilinear",
                              align_corners=False)[0]
        wins[j] = t.numpy()
    x = torch.from_numpy(wins)
    if norm == "imagenet":
        x = (x - _MEAN) / _STD
    return x


_WLUT = {}
def gpu_windows(vol, mask, k, res, dev, norm=NORM, mirror=False):
    """Same windows as eval_windows, built on the GPU from the uint8 volume. Values come from a per-channel
    256-entry table computed on the CPU with eval_windows' own float32 ops, so they are bit-identical."""
    D = vol.shape[0]
    cs = np.array([max(1, min(c, D - 2)) for c in _eval_centers(mask, D, k)])
    tri = torch.from_numpy(np.ascontiguousarray(vol[np.stack([cs - 1, cs, cs + 1], 1)])).to(dev)   # (k,3,IMG,IMG) uint8
    if mirror:
        tri = _mirror_tri(tri, cs, D)
    assert tri.shape[-2:] == (IMG, IMG) and res == INPUT_RES
    lo = (IMG - res) // 2
    tri = tri[:, :, lo:lo + res, lo:lo + res]
    key = (str(dev), norm)
    if key not in _WLUT:
        v = torch.from_numpy((np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0))
        t = torch.stack([v, v, v], 0).view(3, 256, 1)
        if norm == 'imagenet':
            t = (t - _MEAN.view(3, 1, 1)) / _STD.view(3, 1, 1)
        _WLUT[key] = t.view(3, 256).contiguous().to(dev)
    L = _WLUT[key]
    x = torch.stack([L[c][tri[:, c].long()] for c in range(3)], 1)                   # exact float32 values
    if x.shape[-1] != res:
        if norm == 'imagenet':   # resize must happen BEFORE normalisation, as in eval_windows
            L0 = _WLUT.setdefault((str(dev), 'none'), torch.from_numpy(np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0).to(dev))
            x = L0[tri.long()]
            x = F.interpolate(x, size=(res, res), mode='bilinear', align_corners=False)
            x = (x - _MEAN.to(dev)) / _STD.to(dev)
        else:
            x = F.interpolate(x, size=(res, res), mode='bilinear', align_corners=False)
    return x


@torch.no_grad()
def infer_probs(model, xwins, device):
    assert str(device).startswith("cuda"), "Qualified inference requires CUDA/fp16"
    x = xwins.unsqueeze(0).to(device)
    with torch.autocast("cuda", dtype=torch.float16):
        logits = model(x).float()
        assert torch.isfinite(logits).all(), "Nonfinite fp16 logits"
        o = torch.sigmoid(logits)
    return o[0].cpu().numpy()


def rankpct(x):                                   # per-column percentile rank in [0,1]
    order = x.argsort(0).argsort(0).astype(np.float64)
    return order / max(1, (x.shape[0] - 1))


# ============================================================================
# Preprocessing -- verbatim from kprep2/dino_preprocess.py, retargeted to TEST
# ============================================================================
def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    def _hdr(f):
        try:
            h = pydicom.dcmread(f, stop_before_pixels=True)
            iop = getattr(h, 'ImageOrientationPatient', None)
            ipp = getattr(h, 'ImagePositionPatient', None)
            if iop is not None and ipp is not None and len(iop) == 6:
                r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
            else:
                pos = float(getattr(h, 'InstanceNumber', 0) or 0)
            ps = getattr(h, 'PixelSpacing', None); ps = float(ps[0]) if ps is not None else 0.5
            return (pos, f, ps, True)
        except Exception:
            return (0.0, f, 0.5, False)

    def order_and_meta(sdir):
        # Header reads are latency-bound on Kaggle's mounted competition data (~240k opens on the hidden
        # set); a thread pool hides that latency. Records and sort are unchanged -> identical ordering.
        fs = glob.glob(sdir + "/*.dcm")
        with ThreadPoolExecutor(max_workers=HDR_THREADS) as ex:
            out = list(ex.map(_hdr, fs))
        recs = [(pos, f, ps) for pos, f, ps, _ in out]; ps_list = [ps for _, _, ps, ok in out if ok]
        recs.sort(key=lambda x: x[0])
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return [(f, ps) for _, f, ps in recs], med_ps

    def read_px(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))
        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)

    return order_and_meta, read_px, mm_crop_resize


def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r['Anatomical_Plane'] == plane and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get('Fluid_Sensitive', 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None


def build_study(sid, ser_records, tsdir, reader):
    order_and_meta, read_px, mm_crop_resize = reader
    rows = ser_records.get(sid, [])
    vol = np.zeros((MAXS, IMG, IMG), np.uint8); idx = 0; used = set()
    for plane, fluid, k in SLOTS:
        r = _pick_series_for_slot(rows, plane, fluid, used)
        if r is None:
            idx += k; continue
        used.add(r['SeriesInstanceUID'])
        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r['SeriesInstanceUID']}")
        if not files:
            idx += k; continue
        # wide span: the collateral ligaments and lateral meniscus live in the
        # peripheral slices the old 0.15-0.85 crop threw away. Must match the corpus
        # the weights were trained on (build_corpus80.py, SPAN).
        n = len(files); lo, hi = int(n * 0.02), int(n * 0.98) - 1; hi = max(hi, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k
        sel = [files[min(p, n - 1)] for p in picks]
        def _px(fp_ps):
            fp, ps = fp_ps
            try: return read_px(fp), ps
            except Exception: return None, med_ps
        with ThreadPoolExecutor(max_workers=PX_THREADS) as ex:
            got = list(ex.map(_px, sel))                    # order preserved
        arrs = [a for a, _ in got]; pss = [ps for _, ps in got]
        valid = [a for a in arrs if a is not None]
        if valid:
            allpx = np.concatenate([a.ravel() for a in valid])
            loq, hiq = np.percentile(allpx, [2.0, 98.0])
        else:
            loq, hiq = 0.0, 1.0
        for a, ps in zip(arrs, pss):
            if idx >= MAXS: break
            if a is None: idx += 1; continue
            aw = np.clip((a - loq) / (hiq - loq + 1e-6), 0, 1)
            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)
            vol[idx] = (aw * 255).astype(np.uint8); idx += 1
        if idx >= MAXS: break
    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)
    return vol, mask



# ---- parallel preprocessing helpers (module level so they pickle under fork) ----
_SER = None; _TSDIR = None; _READER = None
def _pre_init():
    global _READER
    _READER = _load_fastread().make_builder(FASTREAD_VARIANT, IMG, SLOTS, hdr_threads=FR_HDR, px_threads=FR_PX)
def _pre_one(sid):
    try:
        vol, mask = _READER(sid, _SER, _TSDIR)
    except Exception as e:
        print(f"  [pre] {sid[:16]} FAILED ({type(e).__name__}: {e})", flush=True)
        vol = np.zeros((MAXS, IMG, IMG), np.uint8); mask = np.zeros(MAXS, np.uint8)
    return sid, vol, mask

# ============================================================================
# Test-root discovery + weights + main
# ============================================================================
def find_test_root():
    cands = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
             "/kaggle/input/rsna-knee-abnormality-detection"]
    for b in cands:
        if os.path.exists(b + "/test.csv"):
            return b
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f and (os.path.isdir(d + "/test_series") or os.path.isdir(d + "/test_images")):
            return d
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f:
            return d
    raise RuntimeError("no test root under /kaggle/input")


def find_weight_file(fname):
    # direct dataset mounts first; NEVER recursive-glob the competitions DICOM tree.
    direct = [f"/kaggle/input/raptor-knee-arms/{fname}",
              f"/kaggle/input/raptor-knee-arms/1/{fname}",
              f"/kaggle/input/raptor-cnn336/{fname}"]
    for p in direct:
        if os.path.exists(p):
            return p
    for d in sorted(glob.glob("/kaggle/input/*/")):
        # skip the competition data wherever Kaggle mounts it (/kaggle/input/competitions/<slug> OR /kaggle/input/<slug>):
        # recursively globbing its DICOM tree hung a commit run for 20+ min (2026-09-26)
        if "competition" in d.lower() or "abnormality-detection" in d or any(
                os.path.exists(os.path.join(d, x)) for x in ("test_series", "train_images", "train.csv", "test.csv")):
            continue
        hits = glob.glob(os.path.join(d, "**", fname), recursive=True)
        if hits:
            return hits[0]
    raise RuntimeError(f"{fname} not found under /kaggle/input")


def main():
    import pandas as pd
    t0 = time.time()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ngpu = torch.cuda.device_count()
    print(f"device {dev} | gpus {ngpu} | torch {torch.__version__}", flush=True)

    ROOT = find_test_root()
    tsdir = ROOT + "/test_series"
    if not os.path.isdir(tsdir):
        tsdir = ROOT + "/test_images"
    print("test root:", ROOT, "| series dir:", tsdir, flush=True)

    test = pd.read_csv(ROOT + "/test.csv"); test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
    test_ids = test["StudyInstanceUID"].tolist()
    tser = pd.read_csv(ROOT + "/test_series.csv")
    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)
    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)
    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}
    print(f"test studies {len(test_ids)} | test series {len(tser)}", flush=True)

    sub_cols = ["StudyInstanceUID"] + LAB
    ssub = os.path.join(ROOT, "sample_submission.csv")
    if os.path.exists(ssub):
        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)

    N = len(test_ids); A = len(ARMS)
    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]
    pos = {sid: i for i, sid in enumerate(test_ids)}

    # ---- 2. GPU consumers: one thread per GPU, every arm loaded on every GPU (per-study batch of 1,
    #         so the math is identical to the sequential kernel; only the wall-clock changes) ----
    xla = None
    try:
        import torch_xla; import torch_xla.core.xla_model as xm
        dev = torch_xla.device(); xla = xm
        print("device: XLA/TPU", dev, flush=True)
    except Exception as e:
        print("XLA unavailable:", type(e).__name__, str(e)[:80], "| device", dev, flush=True)
    devs = [f"cuda:{g}" for g in range(ngpu)] if (xla is None and ngpu > 0) else [dev]
    models = {}
    for d in devs:
        models[d] = []
        for a, arm in enumerate(ARMS):
            model, res = load_model(find_weight_file(arm["file"]), arm["arch"], arm["res"], d)
            models[d].append((model, res))
        print(f"[gpu {d}] loaded {A} arm(s) | {time.time()-t0:.0f}s", flush=True)
    q = queue.Queue(maxsize=QUEUE_MAX); done_n = [0]; lock = threading.Lock()

    def consumer(d):
        while True:
            item = q.get()
            if item is None: q.task_done(); return
            sid, vol, mask = item; i = pos[sid]
            try:
                for a, (model, res) in enumerate(models[d]):
                    xw = eval_windows(vol, mask, k=K_EVAL, res=res, norm=NORM) if xla is not None else gpu_windows(vol, mask, K_EVAL, res, d)
                    assert xla is None, "Anatomical L/R TTA requires CUDA/fp16"
                    xm = gpu_windows(vol, mask, K_EVAL, res, d, norm=NORM, mirror=True)
                    arm_probs[a][i] = infer_pair_probs(model, xw, xm, d)
                    del xm
            except Exception as e:
                print(f"  [gpu {d}] study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})", flush=True)
            with lock:
                done_n[0] += 1
                if done_n[0] % 200 == 0 or done_n[0] == N:
                    print(f"  [gpu] {done_n[0]}/{N} | {time.time()-t0:.0f}s", flush=True)
            q.task_done()

    threads = [threading.Thread(target=consumer, args=(d,), daemon=True) for d in devs]
    for th in threads: th.start()

    # ---- 3. prep every study ONCE in a process pool and stream it to the GPUs as it lands ----
    import multiprocessing as mp
    global _SER, _TSDIR, _READER
    _SER, _TSDIR = SER, tsdir
    nproc = max(4, min(96, (os.cpu_count() or 8) - 8))
    t_pre = time.time()
    with mp.get_context("fork").Pool(nproc, initializer=_pre_init) as pool:
        for i, (sid, vol, mask) in enumerate(pool.imap_unordered(_pre_one, test_ids, chunksize=2)):
            q.put((sid, vol, mask))
            if (i + 1) % 200 == 0 or i + 1 == N:
                print(f"  [pre] {i+1}/{N} | {time.time()-t0:.0f}s", flush=True)
    print(f"[pre] done: {N} studies in {time.time()-t_pre:.0f}s (procs {nproc} x fastread {FASTREAD_VARIANT} {FR_HDR}/{FR_PX} threads)", flush=True)
    for _ in devs: q.put(None)
    q.join()
    for th in threads: th.join()
    print(f"[gpu] done: {done_n[0]}/{N} studies on {len(devs)} device(s) | {time.time()-t0:.0f}s", flush=True)
    for d in devs:
        for m, _ in models[d]: del m
    models.clear(); gc.collect()

    # WEIGHTED rank-mean blend across the test set, per finding (the offline recipe).
    # Weights come from ARMS[*]["w"] and are normalised here, so dropping/adding an arm can
    # never silently change the scale. Falls back to equal weights if none are declared.
    _w = np.array([float(a.get("w", 1.0)) for a in ARMS], dtype=np.float64)
    _w = _w / _w.sum()
    print(f"[blend] weighted rank-mean w={dict(zip([a['file'] for a in ARMS], _w.round(4)))}", flush=True)
    ranks = np.tensordot(_w, np.stack([rankpct(np.clip(p, 0, 1)) for p in arm_probs]),
                         axes=(0, 0))                                          # (N,12) in [0,1]
    if not np.isfinite(ranks).all():
        ranks[~np.isfinite(ranks)] = 0.5

    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)
    sub.insert(0, "StudyInstanceUID", test_ids)
    sub = sub[sub_cols]
    assert list(sub.columns) == sub_cols, "column order drift"
    assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"
    assert np.isfinite(sub[LAB].values).all()
    out = "/kaggle/working/submission.csv"
    sub.to_csv(out, index=False)
    sub.to_csv("/kaggle/working/_sota_0949.csv", index=False)
    print("Saved SOTA 0.949 backup to /kaggle/working/_sota_0949.csv", flush=True)
    print("wrote", out, "|", len(sub), "rows x", len(sub.columns), "cols", flush=True)
    print(sub.head().to_string(index=False), flush=True)
    print(f"DONE {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()

In [ ]:

# ==== Stage B: goodpjw2008 ConvNeXt 2.5D on the hidden test set ====
import os, sys, time, glob
import numpy as np, pandas as pd
from scipy.stats import rankdata
CMP = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
D = None
for _c in ('/kaggle/input/datasets/goodpjw2008/rsna-knee-2-5d-convnext-reader',
           '/kaggle/input/goodpjw2008-rsna-knee-2-5d-convnext-reader'):
    if os.path.isdir(_c):
        D = _c
        break
assert D, 'cnxt dataset not mounted'
sys.path.insert(0, D)
from infer import run as cnxt_run
_cks = sorted(glob.glob(f'{D}/cnxt_v0_fold*.pt'))
print('cnxt ckpts:', [os.path.basename(p) for p in _cks], flush=True)
_t = time.time()
_cnxt = cnxt_run(CMP, 'test', _cks, workers=16, bs=8)
print(f'[cnxt] test infer {time.time()-_t:.0f}s | {_cnxt.shape}', flush=True)
assert np.isfinite(_cnxt.to_numpy()).all(), 'cnxt produced non-finite values'
_cnxt.to_csv('/kaggle/working/arm_cnxt_test.csv')
print('[cnxt] wrote arm_cnxt_test.csv', flush=True)

In [ ]:
import os; os.makedirs('knee_meta', exist_ok=True); print('knee_meta/ ready')

In [ ]:
%%writefile knee_meta/__init__.py
"""RSNA knee abnormality detection: labels, cache, model, training, inference."""
__version__ = "0.1.0"

In [ ]:
%%writefile knee_meta/config.py
"""Meta constants shared by every module and both Kaggle notebooks."""
import os
from pathlib import Path

SEED = 2026

PREPROC = "rgb01"

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]
N_LABELS = len(TARGETS)

PLANES = ("Sagittal", "Coronal", "Axial")
SLOTS = [(p, fs) for p in PLANES for fs in (1, 0)]
SLOT_NAMES = ["{}_{}".format(p.upper(), "FLUID" if fs else "STRUCT") for p, fs in SLOTS]
N_SLOT = len(SLOTS)

CROP_MM = 130.0
CACHE_IMG = 336
IMG_SIZES = (336, 224)
TRAIN_IMG_SIZE = 224
EVAL_IMG_SIZE = 224
SLICES_PER_SLOT = 3
SLICE_BAND = (0.20, 0.80)
PLANE_BAND = {
    "Sagittal": (0.12, 0.88),
    "Coronal": (0.20, 0.80),
    "Axial": (0.12, 0.88),
}
LAT_MIN_OFFSET_MM = 20.0

POOL = "cls_mean_focal"
POOL_PARTS = {"cls_mean": 2, "cls_mean_focal": 3}
SLOT_PRIOR_TABLE = {
    "ACL": (0, 3, 5),
    "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4),
    "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5),
    "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5),
    "Effusion": (0, 2),
    "Synovitis": (0, 2),
    "Baker's": (0,),
    "Contusion": (0, 1, 2),
    "Fracture": (0, 1, 2, 4, 5),
}
SLOT_PRIOR_STRENGTH = 0.55

EPOCHS = 10
BATCH_STUDIES = 8
LR_HEAD = 1e-3
LR_BACKBONE = 8e-6
WEIGHT_DECAY = 0.02
UNFREEZE_LAST = 6
WARMUP_STEPS = 100
PAIRWISE_W = 0.10
GOLD_WEIGHT = 8.0
MISSING_CONF = 0.15
MENTION_CONF = 0.70
POSITIVE_CONF = 0.95

AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.10

N_FOLDS = 5
EVAL_BATCH = 8
TIME_BUDGET_S = 8.0 * 3600
HDR_THREADS = 16
PIX_THREADS = 12

DINOV2_CANDIDATES = (
    "/kaggle/input/models/metaresearch/dinov2/pytorch/small/v1",
    "/kaggle/input/models/metaresearch/dinov2/pytorch/small/1",
    "/kaggle/input/dinov2-small",
)

COMPETITION_SLUG = "rsna-knee-abnormality-detection"
LABEL_FILE = "report_labels.csv"


def competition_roots():
    return [
        Path("/kaggle/input/competitions") / COMPETITION_SLUG,
        Path("/kaggle/input") / COMPETITION_SLUG,
        Path("data"),
        Path("."),
    ]


def find_root():
    for cand in competition_roots():
        if (cand / "test.csv").is_file() and (cand / "test_series").is_dir():
            return cand
    base = Path("/kaggle/input")
    if base.is_dir():
        for d1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for d2 in [d1] + sorted(p for p in d1.iterdir() if p.is_dir()):
                if (d2 / "test.csv").is_file() and (d2 / "test_series").is_dir():
                    return d2
    raise FileNotFoundError("competition mount not found")


def find_backbone():
    for cand in DINOV2_CANDIDATES:
        if Path(cand).is_dir() and (Path(cand) / "config.json").is_file():
            return cand
    base = Path("/kaggle/input")
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            if "config.json" in files and "dinov2" in root.lower():
                return root
    return None

In [ ]:
%%writefile knee_meta/dataset.py
"""Study bags from the uint8 cache, with rigid jitter and no laterality flips."""
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

from .config import (AUG_INTENSITY, AUG_ROT_DEG, AUG_SCALE, AUG_SHIFT, N_SLOT,
                     SLICES_PER_SLOT, TARGETS)
from .labels import targets_from_table


def slice_groups(k=SLICES_PER_SLOT):
    return max(1, k // 3)


def pick_group(k, rng, group=None):
    n = slice_groups(k)
    if group is None:
        group = int(rng.integers(n)) if n > 1 else 0
    start = group * 3
    return start, min(start + 3, k)


def augment(x, rng):
    if AUG_ROT_DEG <= 0 and AUG_SCALE <= 0 and AUG_SHIFT <= 0 and AUG_INTENSITY <= 0:
        return x
    h, w = x.shape[-2:]
    angle = float(rng.uniform(-AUG_ROT_DEG, AUG_ROT_DEG)) if AUG_ROT_DEG else 0.0
    scale = float(rng.uniform(1.0 - AUG_SCALE, 1.0 + AUG_SCALE)) if AUG_SCALE else 1.0
    tx = float(rng.uniform(-AUG_SHIFT, AUG_SHIFT)) if AUG_SHIFT else 0.0
    ty = float(rng.uniform(-AUG_SHIFT, AUG_SHIFT)) if AUG_SHIFT else 0.0
    rad = np.deg2rad(angle)
    cos, sin = np.cos(rad) / scale, np.sin(rad) / scale
    theta = torch.tensor([[cos, -sin, tx], [sin, cos, ty]], dtype=x.dtype)
    theta = theta.unsqueeze(0).repeat(x.shape[0], 1, 1)
    grid = F.affine_grid(theta, size=x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border",
                      align_corners=False)
    if AUG_INTENSITY:
        gain = float(rng.uniform(1.0 - AUG_INTENSITY, 1.0 + AUG_INTENSITY))
        bias = float(rng.uniform(-AUG_INTENSITY, AUG_INTENSITY)) * 0.1
        x = x * gain + bias
    return x


class StudyDataset(Dataset):
    def __init__(self, cache, mask, studies, table, train=True, groups=None,
                 img_size=None, study_ids=None):
        self.cache = cache
        self.mask = torch.as_tensor(mask, dtype=torch.float32)
        self.studies = list(studies)
        self.train = train
        self.groups = groups
        self.img_size = img_size
        if study_ids is None:
            order = {s: i for i, s in enumerate(self.studies)}
            table = table[table.StudyInstanceUID.isin(order)]
            rows = table.set_index("StudyInstanceUID").loc[
                [s for s in self.studies if s in set(table.StudyInstanceUID)]]
            values, conf, gold = targets_from_table(rows.reset_index())
            self.uids = list(rows.index)
            self.idx = [order[u] for u in self.uids]
        else:
            pos = {s: i for i, s in enumerate(self.studies)}
            self.uids = [s for s in study_ids if s in pos]
            self.idx = [pos[s] for s in self.uids]
            sub = table.set_index("StudyInstanceUID").loc[self.uids].reset_index()
            values, conf, gold = targets_from_table(sub)
        self.values = torch.as_tensor(values, dtype=torch.float32)
        self.conf = torch.as_tensor(conf, dtype=torch.float32)
        self.gold = torch.as_tensor(gold, dtype=torch.float32)
        self.n_groups = slice_groups(cache.shape[2])

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        row = self.cache[self.idx[i]]
        mask = self.mask[self.idx[i]]
        if self.groups is None:
            group = None
        elif isinstance(self.groups, int):
            group = self.groups
        else:
            group = self.groups[i]
        start, _ = pick_group(row.shape[1], np.random.default_rng(None), group)
        imgs = torch.from_numpy(np.ascontiguousarray(row[:, start:start + 3]))
        if imgs.shape[1] < 3:
            pad = imgs[:, -1:].repeat(1, 3 - imgs.shape[1], 1, 1)
            imgs = torch.cat([imgs, pad], dim=1)
        imgs = imgs.float() / 255.0
        if self.train:
            imgs = augment(imgs, np.random.default_rng(None))
        return {
            "imgs": imgs,
            "mask": mask.clone(),
            "target": self.values[i],
            "conf": self.conf[i],
            "gold": self.gold[i],
        }


def make_dataset(cache, mask, studies, table, train=True, **kw):
    return StudyDataset(cache, mask, studies, table, train=train, **kw)


def make_loader(dataset, batch_size, shuffle, num_workers=0):
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size,
                                       shuffle=shuffle, num_workers=num_workers,
                                       drop_last=shuffle and len(dataset) > batch_size)


def empty_batch(batch_size):
    return {
        "imgs": torch.zeros(batch_size, N_SLOT, 3, 0, 0),
        "mask": torch.zeros(batch_size, N_SLOT),
        "target": torch.zeros(batch_size, len(TARGETS)),
        "conf": torch.zeros(batch_size, len(TARGETS)),
        "gold": torch.zeros(batch_size),
    }

In [ ]:
%%writefile knee_meta/imaging.py
"""DICOM to uint8 cache: physical crop, canonical laterality, slot packing."""
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from .config import (CACHE_IMG, HDR_THREADS, LAT_MIN_OFFSET_MM, N_SLOT,
                     PIX_THREADS, PLANE_BAND, SLICE_BAND,
                     SLICES_PER_SLOT, SLOTS, CROP_MM)

try:
    import pydicom
except ImportError:
    pydicom = None


def _require_pydicom():
    if pydicom is None:
        raise ImportError("pydicom is required to read the DICOM corpus")


_SERIES_INDEX = {}


def _build_index(root, split):
    key = (str(root), split)
    if key in _SERIES_INDEX:
        return _SERIES_INDEX[key]
    idx = {}
    base = Path(root) / split
    if base.is_dir():
        for child in sorted(base.iterdir()):
            if not child.is_dir():
                continue
            idx.setdefault(child.name, child)
            for grand in sorted(child.iterdir()):
                if grand.is_dir():
                    idx.setdefault(grand.name, grand)
    _SERIES_INDEX[key] = idx
    return idx


def find_series_dir(root, uid):
    for split in ("train_series", "test_series"):
        hit = _build_index(root, split).get(uid)
        if hit is not None and hit.exists():
            return hit
    return None


def list_files(series_dir, uid):
    if series_dir is None:
        return []
    if series_dir.is_file():
        return [series_dir]
    files = [p for p in series_dir.rglob("*") if p.is_file()]
    files.sort(key=lambda p: p.name)
    return files


def read_headers(files, threads=HDR_THREADS):
    _require_pydicom()

    def one(path):
        try:
            ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
        except Exception:
            return None
        return ds

    with ThreadPoolExecutor(max_workers=threads) as ex:
        out = list(ex.map(one, files))
    return [h for h in out if h is not None]


def _float_seq(value, minimum):
    if value is None:
        return None
    if not isinstance(value, str):
        try:
            out = [float(x) for x in value]
            return out if len(out) >= minimum else None
        except (TypeError, ValueError):
            pass
    text = str(value).strip().strip("[](){}")
    for ch in "\\|,;":
        text = text.replace(ch, " ")
    try:
        out = [float(p) for p in text.split()]
    except ValueError:
        return None
    return out if len(out) >= minimum else None


def _ipp_vec(h, n=3):
    try:
        v = _float_seq(h.ImagePositionPatient, n)
    except Exception:
        return None
    return np.array(v) if v is not None and len(v) >= n else None


def _iop_vec(h, n=6):
    try:
        v = _float_seq(h.ImageOrientationPatient, n)
    except Exception:
        return None
    return np.array(v) if v is not None and len(v) >= n else None


def _pixel_spacing(h):
    try:
        return _float_seq(h.PixelSpacing, 2)
    except Exception:
        return None


def order_slices(headers):
    keys = []
    for i, h in enumerate(headers):
        ipp, iop = _ipp_vec(h), _iop_vec(h)
        if ipp is None or iop is None:
            keys.append((np.inf, i))
            continue
        normal = np.cross(iop[:3], iop[3:6])
        keys.append((float(np.dot(normal, ipp)), i))
    finite = [k for k in keys if np.isfinite(k[0])]
    if len(finite) < max(1, len(keys) // 2):
        return list(range(len(headers)))
    return [i for _, i in sorted(keys, key=lambda t: (t[0], t[1]))]


def laterality(headers):
    cx = []
    for h in headers:
        ipp, iop, ps = _ipp_vec(h), _iop_vec(h), _pixel_spacing(h)
        if ipp is None or iop is None or ps is None:
            continue
        try:
            rows, cols = int(h.Rows), int(h.Columns)
        except Exception:
            continue
        c = ipp + iop[:3] * ps[1] * cols / 2 + iop[3:6] * ps[0] * rows / 2
        cx.append(float(c[0]))
    if not cx:
        return None
    med = float(np.median(cx))
    if abs(med) < LAT_MIN_OFFSET_MM:
        return None
    return "R" if med < 0 else "L"


def crop_resize(arr, ps_rows, ps_cols, out_size, crop_mm=CROP_MM):
    h, w = arr.shape[:2]
    fh, fw = h * ps_rows, w * ps_cols
    if fh >= crop_mm and fw >= crop_mm:
        ch = min(h, int(round(crop_mm / ps_rows)))
        cw = min(w, int(round(crop_mm / ps_cols)))
        y0 = (h - ch) // 2
        x0 = (w - cw) // 2
        arr = arr[y0:y0 + ch, x0:x0 + cw]
        fh, fw = ch * ps_rows, cw * ps_cols
    scale = out_size / max(fh, fw)
    nh, nw = max(1, int(round(fh * scale))), max(1, int(round(fw * scale)))
    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    resized = cv2.resize(arr, (nw, nh), interpolation=interp)
    canvas = np.full((out_size, out_size), int(np.median(resized)), dtype=resized.dtype)
    y0 = (out_size - nh) // 2
    x0 = (out_size - nw) // 2
    canvas[y0:y0 + nh, x0:x0 + nw] = resized
    return canvas


def normalise_window(arr, h):
    slope = float(getattr(h, "RescaleSlope", 1.0) or 1.0)
    inter = float(getattr(h, "RescaleIntercept", 0.0) or 0.0)
    return arr.astype(np.float32) * slope + inter


def _window(h):
    wc = getattr(h, "WindowCenter", None)
    ww = getattr(h, "WindowWidth", None)
    try:
        return float(wc), float(ww)
    except (TypeError, ValueError):
        return None, None


def window_to_uint8(arr, centre=None, width=None):
    if centre is None or width is None or not width:
        lo, hi = np.percentile(arr, [1.0, 99.0])
        if hi <= lo:
            lo, hi = float(arr.min()), float(arr.max() + 1.0)
        centre, width = (lo + hi) / 2.0, max(hi - lo, 1.0)
    lo, hi = centre - width / 2.0, centre + width / 2.0
    scaled = np.clip((arr - lo) / max(hi - lo, 1e-6), 0.0, 1.0)
    return (scaled * 255.0 + 0.5).astype(np.uint8)


def slot_index(plane, fluid):
    for i, (p, fs) in enumerate(SLOTS):
        if p == plane and fs == int(fluid):
            return i
    return -1


def _sample_indices(n, band, count):
    if n <= 0:
        return []
    if count <= 1:
        return [int(round((band[0] + band[1]) / 2 * (n - 1)))]
    lo = band[0] * (n - 1)
    hi = band[1] * (n - 1)
    step = (hi - lo) / (count - 1) if count > 1 else 0.0
    idx = [int(round(lo + i * step)) for i in range(count)]
    return [min(max(i, 0), n - 1) for i in idx]


def build_cache(root, split, img_size=CACHE_IMG, slices_per_slot=SLICES_PER_SLOT,
                threads=PIX_THREADS, progress=None, max_studies=None):
    _require_pydicom()
    root = str(root)
    series_csv = os.path.join(root, f"{split}_series.csv")
    table = pd.read_csv(series_csv, dtype={"StudyInstanceUID": str,
                                           "SeriesInstanceUID": str})
    studies = sorted(table.StudyInstanceUID.unique())
    if max_studies:
        studies = studies[:max_studies]
    index = {uid: i for i, uid in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, slices_per_slot, img_size, img_size),
                     dtype=np.uint8)
    mask = np.zeros((len(studies), N_SLOT), dtype=np.uint8)

    def decode_one(series_row):
        uid = series_row.StudyInstanceUID
        if uid not in index:
            return None
        sdir = find_series_dir(root, series_row.SeriesInstanceUID)
        files = list_files(sdir, series_row.SeriesInstanceUID)
        if not files:
            return None
        headers = read_headers(files)
        if not headers:
            return None
        order = order_slices(headers)
        headers = [headers[i] for i in order]
        files = [files[i] for i in order]
        plane = str(series_row.Anatomical_Plane)
        fluid = int(series_row.Fluid_Sensitive)
        sidx = slot_index(plane, fluid)
        if sidx < 0:
            return None
        band = PLANE_BAND.get(plane, SLICE_BAND)
        want = _sample_indices(len(headers), band, slices_per_slot + 2)
        if len(headers) > slices_per_slot + 2:
            want = want[1:-1][:slices_per_slot]
        side = laterality(headers)
        slices = []
        for k in want:
            h = headers[k]
            try:
                ds = pydicom.dcmread(str(files[k]), force=True)
                arr = normalise_window(ds.pixel_array, h)
            except Exception:
                arr = None
            if arr is None:
                slices = []
                break
            ps = _pixel_spacing(h)
            if ps is None:
                slices = []
                break
            arr = crop_resize(arr, ps[0], ps[1], img_size)
            wc, ww = _window(h)
            arr = window_to_uint8(arr, wc, ww)
            if side == "L":
                arr = np.ascontiguousarray(arr[:, ::-1])
            slices.append(arr)
        if len(slices) != slices_per_slot:
            return None
        return index[uid], sidx, np.stack(slices)

    rows = [r for r in table.itertuples(index=False)
            if r.StudyInstanceUID in index]
    done = 0
    with ThreadPoolExecutor(max_workers=threads) as ex:
        for result in ex.map(decode_one, rows):
            if result is not None:
                i, sidx, arr = result
                cache[i, sidx] = arr
                mask[i, sidx] = 1
            done += 1
            if progress and done % 200 == 0:
                progress(done, len(rows))
    return cache, mask, np.array(studies, dtype=object)


def save_cache(path, cache, mask, studies):
    np.save(path, cache)
    np.savez(path + ".mask.npz", mask=mask, studies=studies.astype(str))


def load_cache(path):
    cache = np.load(path, allow_pickle=False)
    meta = np.load(path + ".mask.npz", allow_pickle=True)
    return cache, meta["mask"], meta["studies"]

In [ ]:
%%writefile knee_meta/infer.py
"""Offline inference: fold and group average, then per-target rank blend."""
import argparse
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from .config import (EVAL_BATCH, EVAL_IMG_SIZE, PREPROC, TARGETS, find_root)
from .imaging import build_cache, load_cache, save_cache
from .metrics import percentile_rank
from .model import build_model

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

MODEL_CANDIDATES = (
    "models",
    "/kaggle/input/rsna-knee-meta-folds-v2",
    "/kaggle/input/datasets/tensorhydra/rsna-knee-meta-folds-v2",
    "/kaggle/working/models",
)


def find_checkpoints(directory):
    p = Path(directory)
    if not p.is_dir():
        return []
    return sorted(p.glob("fold*.pt"))


def resolve_models_dir(preferred=None, candidates=None):
    """Locate fold*.pt checkpoints: explicit dir, then known mount points."""
    env = os.environ.get("RSNA_MODELS_DIR")
    order = [d for d in (env, preferred) if d]
    order += [d for d in (candidates or MODEL_CANDIDATES) if d not in order]
    for d in order:
        if find_checkpoints(d):
            return str(Path(d))
    raise FileNotFoundError(
        "no fold*.pt checkpoints found; looked in: " + ", ".join(order))


def load_models(models_dir, device, backbone=None, unfreeze=None, tiny=False):
    from .config import UNFREEZE_LAST
    paths = sorted(Path(models_dir).glob("fold*.pt"))
    if not paths:
        raise FileNotFoundError(f"no checkpoints in {models_dir}")
    models = []
    for p in paths:
        blob = torch.load(p, map_location="cpu", weights_only=False)
        if blob.get("preproc") != PREPROC:
            raise ValueError(
                f"{p.name}: checkpoint preproc {blob.get('preproc')!r} != "
                f"infer {PREPROC!r}; checkpoint/inference input contract "
                f"mismatch — use matching checkpoints")
        model = build_model(unfreeze_last=unfreeze or UNFREEZE_LAST,
                            source=backbone, img_size=blob.get("img_size"),
                            tiny=tiny)
        model.load_state_dict(blob["state_dict"])
        model.to(device).eval()
        models.append((p.name, model, blob.get("img_size")))
    return models


@torch.no_grad()
def predict(models, cache, mask, device, batch=EVAL_BATCH, groups=None):
    n = cache.shape[0]
    n_groups = max(1, cache.shape[2] // 3)
    groups = list(range(n_groups)) if groups is None else list(groups)
    total = np.zeros((n, len(TARGETS)), dtype=np.float64)
    count = 0
    for _, model, img_size in models:
        for g in groups:
            start = g * 3
            block = cache[:, :, start:start + 3]
            if block.shape[2] < 3:
                pad = np.repeat(block[:, :, -1:], 3 - block.shape[2], axis=2)
                block = np.concatenate([block, pad], axis=2)
            for i in range(0, n, batch):
                x = torch.from_numpy(block[i:i + batch]).float().div(255.0).to(device)
                m = torch.from_numpy(mask[i:i + batch]).float().to(device)
                logits = model(x, m, img_size=img_size)
                total[i:i + batch] += logits.float().cpu().numpy()
            count += 1
    return total / max(count, 1)


def ensure_test_cache(root, img_size, out=None):
    out = Path(out) if out else Path(root) / f"cache_test_{img_size}.npy"
    if out.is_file() and Path(str(out) + ".mask.npz").is_file():
        return load_cache(str(out))
    cache, mask, studies = build_cache(str(root), "test", img_size=img_size,
                                       progress=_progress("test"))
    save_cache(str(out), cache, mask, studies)
    return cache, mask, studies


def _progress(tag):
    if tqdm is None:
        return None
    bar = {"n": 0}

    def hook(done, total):
        if bar["n"] == 0:
            bar["n"] = 1
            bar["bar"] = tqdm(total=total, desc=tag)
        bar["bar"].n = done
        bar["bar"].refresh()
    return hook


def load_template(root):
    """Study order comes from test.csv (guaranteed replaced at scoring).

    sample_submission.csv is only cross-checked: the competition docs list
    test.csv / test_series.csv / test_series/ as the swapped files, so a
    template built from anything else can silently shrink to the 3-row
    example during scoring.
    """
    root = Path(root)
    frames = {}
    for name in ("test.csv", "sample_submission.csv", "submission.csv"):
        p = root / name
        if p.is_file():
            frames[name] = pd.read_csv(p, dtype={"StudyInstanceUID": str})
    if "test.csv" in frames:
        uids = frames["test.csv"].StudyInstanceUID
        source = "test.csv"
    elif "sample_submission.csv" in frames:
        uids = frames["sample_submission.csv"].StudyInstanceUID
        source = "sample_submission.csv"
    elif "submission.csv" in frames:
        uids = frames["submission.csv"].StudyInstanceUID
        source = "submission.csv"
    else:
        raise FileNotFoundError(f"no submission template or test.csv in {root}")
    sample = frames.get("sample_submission.csv")
    if source != "sample_submission.csv" and sample is not None:
        extra = set(uids) - set(sample.StudyInstanceUID)
        dropped = set(sample.StudyInstanceUID) - set(uids)
        if extra or dropped:
            print(f"template cross-check: {source} has {len(uids)} studies, "
                  f"sample_submission has {len(sample)} "
                  f"(+{len(extra)}/-{len(dropped)}); using {source}",
                  flush=True)
    print(f"template: {len(uids)} studies from {source}", flush=True)
    sub = pd.DataFrame({"StudyInstanceUID": list(uids)})
    for lab in TARGETS:
        sub[lab] = 0.5
    return sub


def check_coverage(sub, studies, max_frac=0.02):
    """Template vs cache coverage; raises when most studies would stay 0.5."""
    known = set(studies)
    missing = int(sum(1 for s in sub.StudyInstanceUID if s not in known))
    frac = missing / max(len(sub), 1)
    print(f"coverage: template {len(sub)}, cache {len(studies)}, "
          f"missing {missing} ({frac:.1%})", flush=True)
    if missing and len(sub) > 10 and frac > max_frac:
        raise ValueError(
            f"{missing}/{len(sub)} template studies missing from cache "
            f"({frac:.1%} > {max_frac:.0%}); refusing to submit a "
            f"0.5-filled file")
    if missing and len(sub) <= 10:
        print(f"warning: {missing} template studies missing from cache "
              f"(kept 0.5)", flush=True)
    return missing


def fill_submission(sub, scores, studies, max_frac=0.02):
    """Write scores into the template; small gaps keep 0.5, big gaps raise."""
    if scores.shape != (len(studies), len(TARGETS)):
        raise ValueError(f"scores {scores.shape} does not match "
                         f"{len(studies)} studies x {len(TARGETS)} targets")
    order = {s: i for i, s in enumerate(studies)}
    rows = np.array([order.get(s, -1) for s in sub.StudyInstanceUID])
    known = rows >= 0
    for j, lab in enumerate(TARGETS):
        col = np.full(len(sub), 0.5)
        col[known] = scores[rows[known], j]
        sub[lab] = col
    check_coverage(sub, studies, max_frac=max_frac)
    if len(sub) > 10:
        flat = [lab for lab in TARGETS if (sub[lab] == 0.5).all()]
        if flat:
            raise ValueError(
                f"constant 0.5 column(s) {flat}: predictions carry no "
                f"ranking, the fallback path fired")
    return sub


def check_slots(mask, studies, min_fill=0.5, max_empty_frac=0.02):
    """Decode health: every study enters the cache even when DICOM decode
    fails, so empty slots are the only signal that predictions would be
    constant ties."""
    fill = float(mask.mean()) if mask.size else 0.0
    empty = int((mask.sum(axis=1) == 0).sum()) if mask.size else 0
    n = len(studies)
    print(f"slots: fill {fill:.1%}, studies with no decoded slots "
          f"{empty}/{n}", flush=True)
    if n > 10 and (fill < min_fill or empty > max_empty_frac * n):
        raise ValueError(
            f"slot decode failed: fill {fill:.1%} (< {min_fill:.0%}) or "
            f"{empty}/{n} studies with no slots (> {max_empty_frac:.0%}) — "
            f"predictions would be mostly ties")
    return fill, empty


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default=None)
    ap.add_argument("--cache", default=None)
    ap.add_argument("--models", default="models")
    ap.add_argument("--out", default="submission.csv")
    ap.add_argument("--batch", type=int, default=EVAL_BATCH)
    ap.add_argument("--img-size", type=int, default=None)
    ap.add_argument("--backbone", default=None)
    ap.add_argument("--groups", default=None)
    ap.add_argument("--device", default=None)
    ap.add_argument("--tiny", action="store_true")
    ap.add_argument("--rank", action="store_true", default=True)
    args = ap.parse_args(argv)

    root = Path(args.root) if args.root else find_root()
    device = torch.device(args.device or ("cuda" if torch.cuda.is_available()
                                          else "cpu"))
    models_dir = resolve_models_dir(args.models)
    print(f"checkpoints: {models_dir}", flush=True)
    models = load_models(models_dir, device, backbone=args.backbone,
                         tiny=args.tiny)
    img_size = args.img_size or models[0][2] or EVAL_IMG_SIZE

    if args.cache:
        cache, mask, studies = load_cache(args.cache)
    else:
        cache, mask, studies = ensure_test_cache(root, img_size)
    studies = [str(s) for s in studies]
    check_slots(mask, studies)
    sub = load_template(root)
    check_coverage(sub, studies)
    groups = None if args.groups is None else [int(g) for g in
                                                args.groups.split(",")]
    print(f"test cache {tuple(cache.shape)} {len(models)} models", flush=True)
    scores = predict(models, cache, mask, device, batch=args.batch, groups=groups)
    if args.rank:
        scores = percentile_rank(scores)

    fill_submission(sub, scores, studies)
    sub.to_csv(args.out, index=False)
    print(f"wrote {args.out}: {len(sub)} studies, {len(TARGETS)} targets",
          flush=True)
    return sub


if __name__ == "__main__":
    main()

In [ ]:
%%writefile knee_meta/labels.py
"""Multilingual rule extraction of the twelve findings from free-text reports.

Every label comes back as (value, confidence). A report that never mentions a
finding is an unknown, not a negative, so it carries a low confidence and pulls
weakly on that output rather than asserting absence.
"""
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

from .config import (LABEL_FILE, MENTION_CONF, MISSING_CONF, N_LABELS,
                     POSITIVE_CONF, TARGETS)

_CHAR_MAP = str.maketrans({"ı": "i", "İ": "i", "ß": "ss", "ø": "o", "Ø": "o",
                           "đ": "d", "Đ": "d"})

TEAR = [
    "tear", "torn", "tearing", "rupture", "ruptured", "disruption", "discontinuity",
    "rotura", "ruptura", "desgarro", "roto", "rota",
    "dechirure", "dechire", "lesion meniscale",
    "scheur", "ruptuur", "gescheurd",
    "riss", "rissbildung", "ruptur", "zerreissung", "einriss",
    "yirtik", "yirtig", "kopma", "butunluk kaybi",
    "ρηξη", "ρηξις", "ρηγμα",
    "руптура", "разкъсв", "разрив", "puknuce", "prekid",
]
SPRAIN = [
    "sprain", "esguince", "entorse", "verstauchung", "verstuiking",
    "distorsiyon", "burkulma", "διαστρεμμα", "навяхван",
    "injury", "lesion", "letsel", "verletzung", "zedelenme",
]
OA_FIND = [
    "osteoarthritis", "osteoarthrosis", "arthrosis", "arthritic",
    "artrosis", "artrose", "arthrose", "gonartrose", "gonartroz", "gonarthrose",
    "gonartro", "artroz",
    "chondrosis", "chondral", "chondropathy", "chondropathie", "chondropatie",
    "chondromalacia", "kondromalazi", "condropatia", "condral", "chondropatia",
    "kraakbeenlijden", "kraakbeenverlies", "kraakbeenschade",
    "knorpelschaden", "knorpeldefekt", "knorpelverlust", "knorpelbelag",
    "cartilage loss", "cartilage thinning", "cartilage fissuring",
    "cartilage defect", "cartilage fissure", "chondral defect", "chondral loss",
    "osteophyt", "osteofit", "osteofyt", "osteophyte", "spurring", "osteofyte",
    "joint space narrowing", "gelenkspaltverschmalerung",
    "kikirdak kaybi", "kikirdak incelme", "kondral",
    "ulcera condral", "hrskavice", "denudacija",
    "χονδροπαθ", "οστεοαρθρ", "οστεοφυτ", "αρθριτ",
    "артроз", "хондропат", "остеофит", "хрущялн",
]
MEDIAL_Q = ["medial", "interno", "interna", "interne", "mediaal", "mediale",
            "innen", "medyal", "medijaln", "εσω", "медиал", "вътреш"]
LATERAL_Q = ["lateral", "externo", "externa", "externe", "buiten", "aussen",
             "lateraln", "εξω", "латерал", "външ"]
PF_Q = ["patellofemoral", "patelofemoral", "femoropatellar", "femoropatelar",
        "femoropatellair", "retropatellar", "retrorotulian", "patellar facet",
        "trochlea", "troclea", "trochlee", "rotula", "rotulian", "patella",
        "patellaire", "patellar", "diz kapagi", "patelofemoraln",
        "επιγονατιδ", "τροχιλ", "пател", "ретропател"]
TRICOMP = ["tricompartmental", "tricompartimental", "three compartments",
           "three compartmens", "all compartments", "gonartrose", "gonartroz",
           "gonarthrose", "gonartrosis", "gonartro", "pangonartro"]

SPECIFIC = {
    "ACL": ["acl", "anterior cruciate", "lca", "ligamento cruzado anterior",
            "ligament croise anterieur", "croise anterieur",
            "voorste kruisband", "vkb", "vorderes kreuzband", "vorderen kreuzband",
            "vordere kreuzband", "on capraz bag", "anterior capraz bag",
            "προσθιο χιαστ", "προσθιου χιαστ", "προσθιος χιαστ",
            "предна кръстна", "предната кръстна", "prednji ukrizeni"],
    "MCL": ["mcl", "medial collateral", "lcm", "ligamento colateral medial",
            "ligamento colateral interno", "ligament collateral medial",
            "collateral medial", "mediale collaterale", "mediaal collateraal",
            "innenband", "mediales kollateralband", "medialen kollateralband",
            "mediale kollateralband", "medyal kollateral", "ic yan bag",
            "εσω πλαγιο", "медиален колатерален", "вътрешна колатерална"],
    "Medial Meniscus": [
        "medial meniscus", "meniscus medialis", "menisco medial", "menisco interno",
        "mediale meniscus", "binnenmeniscus", "innenmeniskus", "meniskus medialis",
        "menisque interne", "menisque medial", "medial menisk", "medyal menisk",
        "εσω μηνισκ", "медиалния менискус", "медиален мениск", "вътрешния мениск",
        "medial and lateral menisc", "medial ve lateral menisk",
        "menisco medial y lateral", "menisco interno y externo",
        "menisco interno y lateral", "mediale en laterale meniscus",
        "innen und aussenmeniskus", "medijalnog meniskusa"],
    "Lateral Meniscus": [
        "lateral meniscus", "meniscus lateralis", "menisco lateral",
        "menisco externo", "laterale meniscus", "buitenmeniscus",
        "aussenmeniskus", "meniskus lateralis", "menisque externe",
        "menisque lateral", "lateral menisk", "εξω μηνισκ",
        "латералния менискус", "латерален мениск", "външния мениск",
        "medial and lateral menisc", "medial ve lateral menisk",
        "menisco medial y lateral", "menisco interno y externo",
        "menisco interno y lateral", "mediale en laterale meniscus",
        "innen und aussenmeniskus", "lateralnog meniskusa"],
}
GENERIC = {
    "ACL": ["cruciate", "cruzado", "croise", "kruisband", "kreuzband",
            "χιαστ", "кръстн", "capraz bag", "ukrizen"],
    "MCL": ["collateral", "colateral", "kollateral", "collaterale",
            "πλαγι", "колатерал", "yan bag"],
    "Medial Meniscus": ["menisc", "menisk", "μηνισκ", "мениск"],
    "Lateral Meniscus": ["menisc", "menisk", "μηνισκ", "мениск"],
}
SIDE_Q = {
    "ACL": ["anterior", "anterieur", "voorste", "vorder", "προσθι", "предн", "prednj"],
    "MCL": MEDIAL_Q,
    "Medial Meniscus": MEDIAL_Q,
    "Lateral Meniscus": LATERAL_Q,
}
DIRECT = {
    "Effusion": [
        "effusion", "joint fluid", "hemarthrosis", "haemarthrosis", "hydrops",
        "derrame", "epanchement", "gewrichtsvocht", "vocht in het gewricht",
        "erguss", "gelenkerguss", "gelenkserguss",
        "eklem ici sivi", "sivi artisi", "efuzyon", "eklem mesafesinde sivi",
        "eklem ici serbest sivi", "eklem sivisi",
        "ενδαρθρικ", "αρθρικο υγρο", "συλλογη υγρου",
        "ставен излив", "излив", "хидропс", "zglobni izljev", "izljev"],
    "Synovitis": [
        "synovitis", "synovial thickening", "synovial hypertrophy",
        "thickened synovial", "hypertrophy of the synovium",
        "synovial proliferation", "proliferation of the synovium",
        "sinovitis", "synovite", "synovitiden", "synovialitis",
        "verdikking van het synovium", "verdikkingen van het synovium",
        "synoviale verdikking", "synovialisverdickung", "synovialverdickung",
        "sinovit", "hoffitis", "sinovyal kalinlasma", "sinovyal proliferasyon",
        "συνοβιτ", "υμενιτ", "синовит", "sinovij", "sinovije"],
    "Baker's": [
        "baker", "popliteal cyst", "poplitealcyst", "popliteal cysts",
        "quiste popliteo", "quistes popliteos", "quiste de baker",
        "kyste de baker", "kyste poplite", "popliteale cyste", "popliteale cyst",
        "bakercyste", "bakerzyste", "poplitealzyste", "popliteazyste",
        "baker kisti", "popliteal kist",
        "κυστη baker", "κυστη του baker", "киста на бейкър",
        "бейкърова киста", "poplitealna cista"],
    "Contusion": [
        "contusion", "contusiones", "contusie", "kontusyon", "kontuzyon",
        "bone bruise", "bone bruising", "knochenprellung", "prellung",
        "botcontusie", "μωλωπ", "θλαση", "контузи", "kontuzij", "nagnjecen"],
    "Fracture": [
        "fracture", "fractur", "fractura", "fraktur", "fractuur", "breuk",
        "kirik", "kirig", "καταγμα", "καταγματ", "фрактура", "счупван",
        "avulsion", "avulsie", "avulsiyon", "prijelom"],
}
NEG_SRC = [
    r"\bno\b", r"\bnot\b", r"\bnon\b", r"\bwithout\b", r"\babsen\w*",
    r"\bnegative for\b", r"\bunremarkable\b", r"\bintact\w*", r"\bnormal\w*",
    r"\bpreserved\b", r"\bfree of\b", r"\bexcluded\b",
    r"\bno hay\b", r"\bsin\b", r"\bausen\w*", r"\bno se\b", r"\bconservad\w*",
    r"\bintegr\w*", r"\bindemne\b", r"\bpas de\b", r"\baucun\w*", r"\bsans\b",
    r"\bgeen\b", r"\bzonder\b", r"\bonopvallend\w*", r"\bnormaal\b",
    r"\bvrij van\b", r"\bintacte\b",
    r"\bkein\w*", r"\bohne\b", r"\bunauffallig\w*", r"\bregelrecht\w*",
    r"\bnicht\b", r"\bintakt\w*",
    r"\byok\w*", r"\bizlenmemis\w*", r"\bizlenmedi\w*", r"\bsaptanmamis\w*",
    r"\bgozlenmemis\w*", r"\bgorulmemis\w*", r"\bkorunmus\w*",
    r"\bmevcut degil\b", r"\bnormaldir\b", r"\bdogaldir\b",
    r"\bδεν\b", r"\bχωρις\b", r"\bφυσιολογικ\w*", r"\bακεραι\w*",
    r"\bбез\b", r"\bняма\b", r"\bне се\b", r"\bнормал\w*", r"\bзапазен\w*",
    r"\bсъхранен\w*", r"\bsenza\b", r"\bsem\b", r"\bnao\b", r"\bbez\b",
    r"\buredn\w*",
]

NEG_RE = re.compile("|".join(NEG_SRC))
NEG_BACK = 70
NEG_FWD = 45
QUAL_WINDOW = 90
PAIR_WINDOW = 160

TRAUMA_SRC = [
    r"\bfracture\w*", r"\bfractur\w*", r"\bfraktur\w*", r"\bfractuur\w*",
    r"\bbreuk\w*", r"\bkirik\w*", r"\bavuls\w*", r"\bimpaction\w*",
    r"\bcontus\w*", r"\bcontusion\w*", r"\bbruise\w*", r"\bbruising\b",
    r"\bkontuz\w*", r"\bprellung\w*", r"\bkwetsing\w*", r"\btrauma\w*",
    r"\bakut\w*", r"\bacute\w*", r"\bagud\w*", r"\boedem\w*", r"\bedem\w*",
    r"\bödem\w*", r"\bodem\w*", r"\bedema\w*", r"\bedematoz\b",
    r"\bödematöz\b",
    r"\bκαταγμ\w*", r"\bфрактур\w*", r"\bконтуз\w*", r"\bneglect\w*",
    r"\bsubchondral fracture\w*", r"\bosteochondral injur\w*",
]
TRAUMA_RE = re.compile("|".join(TRAUMA_SRC))
CLAUSE_SPLIT = re.compile(r"[.;:\n\r•·]+|\s-\s|\s>\s|\s\*\s")
PF_MASK = re.compile(
    r"(?:medial|lateral)\s+(?:patellar|patella|facet|faset|trochlea|"
    r"trochlear|retinac)\w*"
    r"|(?:medial|lateral)\w*\s+(?:faset|facet|patel|patell|trochl|retinac)\w*"
    r"|(?:patellar|patella|facet|faset|trochlea|trochlear|retinac|rotul|"
    r"patelo)\w*\s+(?:medial|lateral|superior|inferior)"
    r"|(?:patellofemoral|patelofemoral|femoropatellar|femoropatelar|"
    r"retinacul\w*|retinakul\w*|rotulian\w*|patellar|patella)")

STATUS_NONE = 0
STATUS_NEG = 1
STATUS_POS = 2


def fold_text(text):
    t = str(text).lower().translate(_CHAR_MAP)
    t = unicodedata.normalize("NFKD", t)
    return "".join(c for c in t if not unicodedata.combining(c))


def _find_any(clause, terms):
    best = -1
    for term in terms:
        i = clause.find(term)
        if i >= 0 and (best < 0 or i < best):
            best = i
    return best


def _negated(clause, pos, span):
    back = clause[max(0, pos - NEG_BACK):pos]
    fwd = clause[pos + span:pos + span + NEG_FWD]
    return bool(NEG_RE.search(back) or NEG_RE.search(fwd))


def _direct_status(clause, terms):
    seen = False
    for term in terms:
        start = 0
        while True:
            i = clause.find(term, start)
            if i < 0:
                break
            seen = True
            if not _negated(clause, i, len(term)):
                return STATUS_POS
            start = i + 1
    return STATUS_NEG if seen else STATUS_NONE


def _anatomy_index(clause, label):
    i = _find_any(clause, SPECIFIC[label])
    if i >= 0:
        return i
    g = _find_any(clause, GENERIC[label])
    if g >= 0 and _find_any(clause, SIDE_Q[label]) >= 0:
        return g
    return -1


def _paired_status(clause, label, finds):
    seen = False
    ai = _anatomy_index(clause, label)
    if ai >= 0:
        for term in finds:
            fi = clause.find(term)
            while fi >= 0:
                if abs(fi - ai) <= PAIR_WINDOW:
                    seen = True
                    if not _negated(clause, fi, len(term)):
                        return STATUS_POS
                fi = clause.find(term, fi + 1)
    return STATUS_NEG if seen else STATUS_NONE


def _bump(best, cur):
    return max(best, cur)


def _compartment_status(clause, qualifiers, mask_pf):
    if TRAUMA_RE.search(clause):
        return STATUS_NONE
    best = STATUS_NONE
    seen = False
    for m in re.finditer("|".join(map(re.escape, OA_FIND)), clause):
        oi = m.start()
        if _negated(clause, oi, 8):
            continue
        seen = True
        win = clause[max(0, oi - QUAL_WINDOW):oi + QUAL_WINDOW]
        if mask_pf:
            win = PF_MASK.sub(" ", win)
        if _find_any(win, qualifiers) >= 0:
            best = STATUS_POS
    if _find_any(clause, TRICOMP) >= 0:
        seen = True
        if not _negated(clause, max(0, _find_any(clause, TRICOMP)), 8):
            best = STATUS_POS
    return best if best == STATUS_POS else (STATUS_NEG if seen else STATUS_NONE)


def extract_status(report):
    out = {lab: STATUS_NONE for lab in TARGETS}
    if not isinstance(report, str) or not report.strip():
        return out
    text = fold_text(report)
    for raw in CLAUSE_SPLIT.split(text):
        clause = raw.strip()
        if len(clause) < 3:
            continue
        for lab, terms in DIRECT.items():
            if out[lab] != STATUS_POS:
                out[lab] = _bump(out[lab], _direct_status(clause, terms))
        tear = TEAR + SPRAIN
        for lab in SPECIFIC:
            if out[lab] == STATUS_POS:
                continue
            finds = tear if lab in ("ACL", "MCL") else TEAR
            out[lab] = _bump(out[lab], _paired_status(clause, lab, finds))
        for lab, quals in (("Medial OA", MEDIAL_Q), ("Lateral OA", LATERAL_Q),
                           ("PF OA", PF_Q)):
            if out[lab] != STATUS_POS:
                out[lab] = _bump(out[lab], _compartment_status(
                    clause, quals, mask_pf=lab != "PF OA"))
    return out


def status_to_target(status):
    value = 1.0 if status == STATUS_POS else 0.0
    if status == STATUS_POS:
        conf = POSITIVE_CONF
    elif status == STATUS_NEG:
        conf = MENTION_CONF
    else:
        conf = MISSING_CONF
    return value, conf


def extract_labels(report):
    status = extract_status(report)
    return {lab: status_to_target(status[lab]) for lab in TARGETS}


def build_label_table(train_csv):
    train = pd.read_csv(train_csv)
    cols = list(train.columns)
    rows = []
    for vals in train.itertuples(index=False, name=None):
        rec = dict(zip(cols, vals))
        labels = extract_labels(rec.get("Report"))
        gold = not pd.isna(rec.get("ACL"))
        row = {"StudyInstanceUID": rec["StudyInstanceUID"], "is_gold": int(gold)}
        for lab in TARGETS:
            value = float(rec[lab]) if gold else labels[lab][0]
            row[lab] = value
            row[lab + "__conf"] = 1.0 if gold else labels[lab][1]
        rows.append(row)
    return pd.DataFrame(rows, columns=["StudyInstanceUID", "is_gold"] +
                        [c for lab in TARGETS for c in (lab, lab + "__conf")])


def targets_from_table(table):
    values = table[TARGETS].to_numpy(dtype=np.float32)
    conf = table[[lab + "__conf" for lab in TARGETS]].to_numpy(dtype=np.float32)
    gold = table["is_gold"].to_numpy(dtype=np.float32)
    return values, conf, gold


def save_label_table(train_csv, out_path):
    table = build_label_table(train_csv)
    table.to_csv(out_path, index=False)
    return table


def load_label_table(path):
    return pd.read_csv(path)


CONSENSUS_FILE = "llm_labels_v4_blend.csv"


def resolve_consensus(spec):
    if spec != "auto":
        p = Path(spec)
        if not p.is_file():
            raise FileNotFoundError(f"consensus labels not found: {spec}")
        return p
    roots = [Path("/kaggle/input"), Path.cwd()]
    hits = []
    for root in roots:
        if root.is_dir():
            hits.extend(sorted(root.glob("**/" + CONSENSUS_FILE)))
    if not hits:
        raise FileNotFoundError(
            f"{CONSENSUS_FILE} not found under /kaggle/input or cwd")
    return hits[0]


def apply_consensus(table, partner_path):
    """Downweight cells where a second labeller disagrees with our values.

    Values are untouched (validation truth stays comparable); only the
    training confidence of disagreement cells drops to MISSING_CONF, and
    gold rows are never altered. Returns (table, stats).
    """
    partner = pd.read_csv(partner_path, dtype={"StudyInstanceUID": str})
    idx = partner.set_index("StudyInstanceUID")
    out = table.copy()
    gold = out["is_gold"].to_numpy() == 1
    conf_cols = [lab + "__conf" for lab in TARGETS]
    conf = out[conf_cols].to_numpy(dtype=np.float64).copy()
    before = conf.copy()
    stats = {"partner": Path(partner_path).name, "n_gold_skipped": int(gold.sum()),
             "down": MISSING_CONF, "per_label": {}}
    for j, lab in enumerate(TARGETS):
        if lab not in idx.columns:
            raise KeyError(f"{lab} missing from {partner_path}")
        vals = out["StudyInstanceUID"].map(idx[lab]).astype(float).to_numpy()
        present = ~np.isnan(vals)
        hard = (vals > 0.5) != (out[lab].to_numpy(dtype=float) > 0.5)
        mask = present & hard & ~gold
        conf[mask, j] = MISSING_CONF
        stats["per_label"][lab] = int(mask.sum())
    for j, lab in enumerate(TARGETS):
        out[lab + "__conf"] = conf[:, j]
    stats["n_down"] = int(np.count_nonzero(conf != before))
    return out, stats


def label_columns():
    return TARGETS + [lab + "__conf" for lab in TARGETS] + ["is_gold"]


def audit(table, train_csv):
    train = pd.read_csv(train_csv)
    gold = train.dropna(subset=["ACL"])
    truth = gold[TARGETS].to_numpy(dtype=np.float64) > 0.5
    rule_rows = []
    for rep in gold.Report:
        st = extract_status(rep)
        rule_rows.append([float(st[lab] == STATUS_POS) for lab in TARGETS])
    rule = np.asarray(rule_rows, dtype=np.float64).reshape(-1, len(TARGETS)) > 0.5
    if truth.size == 0:
        return {"n_gold": 0, "macro_agreement": float("nan"),
                "per_label_agreement": {lab: float("nan") for lab in TARGETS},
                "n_labels": N_LABELS, "label_file": LABEL_FILE}
    per_label = dict(zip(TARGETS, (truth == rule).mean(axis=0).tolist()))
    return {
        "n_gold": int(len(gold)),
        "macro_agreement": float((truth == rule).mean()),
        "per_label_agreement": per_label,
        "n_labels": N_LABELS,
        "label_file": LABEL_FILE,
    }

In [ ]:
%%writefile knee_meta/metrics.py
"""Macro AUC for the competition metric and per-target percentile ranks."""
import numpy as np
from sklearn.metrics import roc_auc_score

from .config import TARGETS


def macro_auc(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    aucs = []
    for i in range(y_true.shape[1]):
        col = y_true[:, i]
        if col.min() == col.max():
            continue
        aucs.append(roc_auc_score(col, y_score[:, i]))
    return float(np.mean(aucs)) if aucs else float("nan")


def per_label_auc(y_true, y_score):
    out = {}
    for i, lab in enumerate(TARGETS):
        col = np.asarray(y_true)[:, i]
        if col.min() == col.max():
            out[lab] = float("nan")
        else:
            out[lab] = float(roc_auc_score(col, np.asarray(y_score)[:, i]))
    return out


def percentile_rank(scores):
    scores = np.asarray(scores, dtype=np.float64)
    out = np.empty_like(scores)
    n = scores.shape[0]
    order = np.argsort(scores, axis=0, kind="mergesort")
    ranks = np.empty((n, scores.shape[1]), dtype=np.float64)
    for c in range(scores.shape[1]):
        col = order[:, c]
        r = np.empty(n, dtype=np.float64)
        r[col] = np.arange(n, dtype=np.float64)
        vals = scores[col, c]
        i = 0
        while i < n:
            j = i
            while j + 1 < n and vals[j + 1] == vals[i]:
                j += 1
            if j > i:
                r[col[i:j + 1]] = 0.5 * (i + j)
            i = j + 1
        ranks[:, c] = r
    out = (ranks + 0.5) / n
    return out

In [ ]:
%%writefile knee_meta/model.py
"""Backbone plus slot-attention head: one bag of slot images to twelve logits."""
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, ViTConfig, ViTModel

from .config import (N_LABELS, N_SLOT, POOL, POOL_PARTS, SLOT_PRIOR_STRENGTH,
                     SLOT_PRIOR_TABLE, TARGETS, UNFREEZE_LAST)


class SlotHead(nn.Module):
    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=True):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == N_SLOT and n_out == N_LABELS:
            for label, slots in SLOT_PRIOR_TABLE.items():
                if label in TARGETS:
                    p_[TARGETS.index(label), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer("slot_prior", p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class Model(nn.Module):
    def __init__(self, backbone, dim, pool=POOL, prior=True, img_size=None):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.img_size = img_size
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, N_LABELS, prior=prior)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def encode(self, x, img_size=None):
        if img_size is not None and img_size != x.shape[-1]:
            x = F.interpolate(x, size=(img_size, img_size), mode="bilinear",
                              align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == "cls_mean_focal":
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        return torch.cat(parts, dim=1)

    def forward(self, imgs, mask, img_size=None):
        if imgs.dtype == torch.uint8:
            raise ValueError(
                "Model.forward expects float images in [0,1], got uint8; "
                "divide by 255 before calling the model")
        lo, hi = float(imgs.min()), float(imgs.max())
        if hi > 1.5 or lo < -0.5:
            raise ValueError(
                f"Model.forward input range [{lo:.3f}, {hi:.3f}] outside "
                f"[-0.5, 1.5]; expected float images in [0,1]")
        b, s = imgs.shape[:2]
        x = imgs.reshape(b * s, *imgs.shape[2:]).float()
        feat = self.encode(x, img_size if img_size is not None else self.img_size)
        return self.head(feat.reshape(b, s, -1), mask)


def _tiny_backbone(img_size=64):
    cfg = ViTConfig(hidden_size=64, num_hidden_layers=2, num_attention_heads=2,
                    intermediate_size=128, image_size=img_size, patch_size=16,
                    num_channels=3)
    return ViTModel(cfg)


def build_model(unfreeze_last=UNFREEZE_LAST, source=None, pool=POOL, prior=True,
                tiny=False, img_size=None):
    if tiny:
        bb = _tiny_backbone(img_size or 64)
        dim = bb.config.hidden_size
        return Model(bb, dim, pool=pool, prior=prior, img_size=img_size)
    if source is None:
        from .config import find_backbone
        source = find_backbone()
    if source is None:
        raise FileNotFoundError("DINOv2 weights not attached")
    bb = AutoModel.from_pretrained(str(source))
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True
    return Model(bb, bb.config.hidden_size, pool=pool, prior=prior, img_size=img_size)


def param_groups(model, lr_head, lr_backbone, weight_decay):
    head = list(model.head.parameters())
    backbone = [p for p in model.backbone.parameters() if p.requires_grad]
    return [
        {"params": head, "lr": lr_head},
        {"params": backbone, "lr": lr_backbone},
    ], {"weight_decay": weight_decay}

In [ ]:
%%writefile knee_meta/train.py
"""Five-fold CV training: confidence-weighted BCE plus a pairwise ranking term."""
import argparse
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.model_selection import KFold

from .config import (BATCH_STUDIES, EPOCHS, EVAL_BATCH, GOLD_WEIGHT, LR_BACKBONE,
                     LR_HEAD, N_FOLDS, PAIRWISE_W, PREPROC, SEED, TIME_BUDGET_S,
                     TRAIN_IMG_SIZE, UNFREEZE_LAST, WEIGHT_DECAY, find_root)
from .dataset import make_dataset, make_loader
from .imaging import load_cache
from .labels import (apply_consensus, audit, load_label_table,
                     resolve_consensus, save_label_table)
from .metrics import macro_auc, per_label_auc
from .model import build_model, param_groups


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def array_stats(arr, target=2_000_000):
    """Mean and std from a strided sample, never a full-array float64 copy."""
    flat = arr.reshape(-1)
    step = max(1, flat.size // target)
    sample = flat[::step]
    return float(sample.mean()), float(sample.std())


def average_states(states):
    """Mean of cloned state dicts; non-float buffers keep the last snapshot."""
    out = {}
    for key in states[0]:
        vals = [s[key] for s in states]
        if torch.is_floating_point(vals[0]):
            out[key] = torch.stack(vals).float().mean(0).to(vals[0].dtype)
        else:
            out[key] = vals[-1].clone()
    return out


def parse_fold_spec(spec):
    """'0' or '0,1,2' -> set of fold ids; None/'' stays None."""
    if spec is None or not str(spec).strip():
        return None
    return {int(x) for x in str(spec).split(",") if str(x).strip()}


def enable_determinism():
    """Strict bitwise-reproducible kernels (CUDA); must run before matmuls."""
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)


def determinism_check(model, batch, device, img_size, seed=1234):
    """Two identical forward/backward passes -> (ok, why, strict_unusable)."""
    state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    cpu_rng = torch.get_rng_state()
    cuda_rng = (torch.cuda.get_rng_state_all()
                if torch.cuda.is_available() else None)

    def run_pass():
        model.load_state_dict(state)
        model.zero_grad(set_to_none=True)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        model.train()
        try:
            _, _, _, _, loss = step(model, batch, device, img_size, train=True)
            loss.backward()
        except RuntimeError as exc:
            model.zero_grad(set_to_none=True)
            return None, str(exc)
        grads = {n: p.grad.detach().clone() for n, p in model.named_parameters()
                 if p.grad is not None}
        model.zero_grad(set_to_none=True)
        return grads, None

    g1, e1 = run_pass()
    g2, e2 = run_pass()
    model.load_state_dict(state)
    model.zero_grad(set_to_none=True)
    torch.set_rng_state(cpu_rng)
    if cuda_rng is not None:
        torch.cuda.set_rng_state_all(cuda_rng)
    if e1 is not None or e2 is not None:
        return False, (e1 or e2)[:300], True
    same = all(torch.allclose(g1[k], g2[k], rtol=0.0, atol=0.0, equal_nan=True)
               for k in g1)
    return same, ("bitwise equal" if same else "gradients differ"), False


def weighted_bce(logits, target, conf, gold):
    per = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    if gold.dim() == 1:
        gold = gold.unsqueeze(1)
    w = conf * (1.0 + (GOLD_WEIGHT - 1.0) * gold)
    return (per * w).sum() / w.sum().clamp_min(1e-6)


def pairwise_loss(logits, target, conf, margin=1.0):
    if PAIRWISE_W <= 0:
        return logits.sum() * 0.0
    sign = target[:, None, :] - target[None, :, :]
    valid = sign != 0
    if not bool(valid.any()):
        return logits.sum() * 0.0
    gap = logits[:, None, :] - logits[None, :, :]
    w = torch.minimum(conf[:, None, :], conf[None, :, :])
    loss = F.relu(margin - torch.sign(sign) * gap)
    return (loss * valid.float() * w).sum() / (valid.float() * w).sum().clamp_min(1e-6)


def step(model, batch, device, img_size, train=True):
    imgs = batch["imgs"].to(device, non_blocking=True)
    mask = batch["mask"].to(device, non_blocking=True)
    target = batch["target"].to(device, non_blocking=True)
    conf = batch["conf"].to(device, non_blocking=True)
    gold = batch["gold"].to(device, non_blocking=True)
    logits = model(imgs, mask, img_size=img_size)
    loss = weighted_bce(logits, target, conf, gold)
    if train:
        loss = loss + PAIRWISE_W * pairwise_loss(logits, target, conf)
    return logits.detach(), target, conf, gold, loss


@torch.no_grad()
def evaluate(model, loader, device, img_size):
    model.eval()
    preds, truths = [], []
    total = 0.0
    n = 0
    for batch in loader:
        logits, target, conf, gold, loss = step(model, batch, device, img_size,
                                                train=False)
        preds.append(logits.cpu().numpy())
        truths.append(target.cpu().numpy())
        total += float(loss) * len(target)
        n += len(target)
    preds = np.concatenate(preds)
    truths = np.concatenate(truths)
    return macro_auc(truths, preds), total / max(n, 1), per_label_auc(truths, preds)


def train_fold(fold, tr_idx, va_idx, studies, cache, mask, table, args, device):
    train_ds = make_dataset(cache, mask, studies, table, train=True,
                            study_ids=[studies[i] for i in tr_idx],
                            img_size=args.img_size)
    val_ds = make_dataset(cache, mask, studies, table, train=False,
                          study_ids=[studies[i] for i in va_idx],
                          img_size=args.img_size)
    train_loader = make_loader(train_ds, args.batch_size, shuffle=True)
    val_loader = make_loader(val_ds, args.eval_batch, shuffle=False)
    model = build_model(unfreeze_last=args.unfreeze, source=args.backbone,
                        img_size=args.img_size, tiny=getattr(args, "tiny", False))
    model.to(device)
    groups, extra = param_groups(model, LR_HEAD, LR_BACKBONE, WEIGHT_DECAY)
    opt = torch.optim.AdamW(groups, **extra)
    if getattr(args, "deterministic", False):
        rng_cpu = torch.get_rng_state()
        rng_cuda = (torch.cuda.get_rng_state_all()
                    if torch.cuda.is_available() else None)
        probe = next(iter(train_loader))
        ok, why, strict_unusable = determinism_check(
            model, probe, device, args.img_size)
        print(f"determinism check: {'PASS' if ok else 'FAIL'} ({why})",
              flush=True)
        if strict_unusable:
            torch.use_deterministic_algorithms(False)
            ok, why, _ = determinism_check(model, probe, device, args.img_size)
            print(f"determinism check (best-effort): "
                  f"{'PASS' if ok else 'FAIL'} ({why})", flush=True)
        torch.set_rng_state(rng_cpu)
        if rng_cuda is not None:
            torch.cuda.set_rng_state_all(rng_cuda)
    total_steps = max(1, args.epochs * len(train_loader))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / max(1, args.warmup)) *
        (0.5 * (1.0 + np.cos(np.pi * min(s, total_steps) / total_steps))))
    best, best_state, history = -1.0, None, []
    swa_states = []
    started = time.time()
    for epoch in range(args.epochs):
        model.train()
        total, n = 0.0, 0
        for batch in train_loader:
            opt.zero_grad(set_to_none=True)
            _, _, _, _, loss = step(model, batch, device, args.img_size, train=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            total += float(loss.detach()) * len(batch["target"])
            n += len(batch["target"])
        val_auc, val_loss, per_label = evaluate(model, val_loader, device,
                                                args.img_size)
        history.append({"epoch": epoch, "train_loss": total / max(n, 1),
                        "val_auc": val_auc, "val_loss": val_loss,
                        "per_label": per_label})
        print(f"fold {fold} epoch {epoch}: train {total / max(n, 1):.4f} "
              f"val_auc {val_auc:.4f}", flush=True)
        if val_auc > best:
            best = val_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if args.swa_from >= 0 and epoch >= args.swa_from:
            swa_states.append({k: v.detach().cpu().clone()
                               for k, v in model.state_dict().items()})
        if time.time() - started > args.budget_per_fold:
            print(f"fold {fold}: budget reached at epoch {epoch}", flush=True)
            break
    swa = {}
    if swa_states:
        swa_state = average_states(swa_states)
        model.load_state_dict(swa_state)
        swa_auc, swa_loss, _ = evaluate(model, val_loader, device, args.img_size)
        swa = {"swa_val_auc": swa_auc, "swa_members": len(swa_states),
               "swa_from": args.swa_from, "swa_selected": swa_auc > best}
        print(f"fold {fold} swa: {swa_auc:.4f} "
              f"({len(swa_states)} members, selected={swa_auc > best})", flush=True)
        if swa_auc > best:
            best = swa_auc
            best_state = swa_state
    elif args.swa_from >= 0:
        print(f"fold {fold}: no swa members (swa_from={args.swa_from})", flush=True)
    if best_state is not None:
        model.load_state_dict(best_state)
    out = Path(args.out) / f"fold{fold}.pt"
    torch.save({"state_dict": model.state_dict(), "val_auc": best,
                "img_size": args.img_size, "fold": fold, "preproc": PREPROC,
                "history": history, **swa}, out)
    return best, history


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default=None)
    ap.add_argument("--cache", default=None)
    ap.add_argument("--labels", default=None)
    ap.add_argument("--out", default="models")
    ap.add_argument("--folds", type=int, default=N_FOLDS)
    ap.add_argument("--epochs", type=int, default=EPOCHS)
    ap.add_argument("--batch-size", type=int, default=BATCH_STUDIES)
    ap.add_argument("--eval-batch", type=int, default=EVAL_BATCH)
    ap.add_argument("--img-size", type=int, default=None)
    ap.add_argument("--unfreeze", type=int, default=UNFREEZE_LAST)
    ap.add_argument("--backbone", default=None)
    ap.add_argument("--warmup", type=int, default=100)
    ap.add_argument("--budget", type=float, default=TIME_BUDGET_S)
    ap.add_argument("--seed", type=int, default=SEED)
    ap.add_argument("--device", default=None)
    ap.add_argument("--tiny", action="store_true")
    ap.add_argument("--only-fold", type=str, default=None,
                    help="train only these folds, '0' or '0,1,2' (split "
                         "sessions for long runs)")
    ap.add_argument("--swa-from", type=int, default=-1,
                    help="average weights from this epoch onward; -1 disables")
    ap.add_argument("--deterministic", action="store_true",
                    help="reproducible kernels (paired screening)")
    ap.add_argument("--consensus", default=None,
                    help="path or 'auto': downweight cells where a second "
                         "labeller disagrees (values unchanged)")
    args = ap.parse_args(argv)
    args.only_fold = parse_fold_spec(args.only_fold)
    args.budget_per_fold = args.budget / max(1, args.folds)
    if args.deterministic:
        enable_determinism()
    seed_everything(args.seed)

    root = Path(args.root) if args.root else find_root()
    cache_path = Path(args.cache) if args.cache else root / "cache_train.npy"
    label_path = Path(args.labels) if args.labels else Path("report_labels.csv")
    Path(args.out).mkdir(parents=True, exist_ok=True)

    if label_path.is_file():
        table = load_label_table(label_path)
    else:
        table = save_label_table(root / "train.csv", label_path)
    print("label audit:", json.dumps(audit(table, root / "train.csv"), indent=2))
    if args.consensus:
        table, cons_stats = apply_consensus(
            table, resolve_consensus(args.consensus))
        print("consensus:", json.dumps(cons_stats), flush=True)

    cache, mask, studies = load_cache(str(cache_path))
    studies = [str(s) for s in studies]
    known = set(table.StudyInstanceUID)
    keep = [i for i, s in enumerate(studies) if s in known]
    if len(keep) != len(studies):
        cache, mask, studies = cache[keep], mask[keep], [studies[i] for i in keep]
    if args.img_size is None:
        args.img_size = TRAIN_IMG_SIZE
    device = torch.device(args.device or ("cuda" if torch.cuda.is_available()
                                          else "cpu"))
    print(f"cache {tuple(cache.shape)} device {device} img {args.img_size}",
          flush=True)
    print("slot fill:", np.round(mask.mean(axis=0), 3).tolist(),
          "| pixel mean %.1f std %.1f" % array_stats(cache), flush=True)

    order = np.random.RandomState(args.seed).permutation(len(studies))
    splits = KFold(n_splits=args.folds, shuffle=False)
    results = []
    started = time.time()
    for fold, (tr, va) in enumerate(splits.split(order)):
        if args.only_fold is not None and fold not in args.only_fold:
            continue
        auc, history = train_fold(fold, order[tr], order[va], studies, cache,
                                  mask, table, args, device)
        results.append({"fold": fold, "val_auc": auc})
        print(f"fold {fold} best val_auc {auc:.4f}", flush=True)
        if time.time() - started > args.budget:
            print("global budget reached", flush=True)
            break
    summary = Path(args.out) / "cv_summary.json"
    summary.write_text(json.dumps(results, indent=2))
    print("cv:", json.dumps(results), flush=True)
    return results


if __name__ == "__main__":
    main()

In [ ]:

# ==== Stage C: knee_meta v4 (ensemble of 5) on the hidden test set ====
import os, sys, time, glob, subprocess as _sp
sys.path.insert(0, os.getcwd())
assert os.path.isfile('knee_meta/infer.py'), 'knee_meta writefiles missing'
_MODELS = None
for _pat in ('/kaggle/input/datasets/tensorhydra/rsna-knee-meta-folds-v4',
             '/kaggle/input/isotriage-rsna-knee-meta-folds-v4',
             '/kaggle/input/datasets/isotriage/rsna-knee-meta-folds-v4'):
    if glob.glob(_pat + '/fold0.pt'):
        _MODELS = _pat
        break
assert _MODELS, 'rsna-knee-meta-folds-v4 not mounted'
print('[ours] models:', _MODELS, flush=True)
_env = {**os.environ, 'PYTHONPATH': os.getcwd(),
        'PYTHONDONTWRITEBYTECODE': '1',
        'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'}
_cache = '/kaggle/working/cache_test_336.npy'
_t = time.time()
_rc = _sp.run(['python', '-c',
              'from knee_meta.config import find_root; from knee_meta.infer import '
              'ensure_test_cache; ensure_test_cache(str(find_root()), 336, out=%r)' % _cache],
             env=_env, cwd=os.getcwd(), timeout=7200).returncode
print(f'[ours] cache rc={_rc} took {(time.time()-_t)/60:.1f} min', flush=True)
assert _rc == 0, f'ours cache rc={_rc}'
_t = time.time()
_rc = _sp.run(['python', '-m', 'knee_meta.infer',
               '--cache', _cache, '--models', _MODELS, '--img-size', '336',
               '--out', '/kaggle/working/arm_ours_test.csv'],
              env=_env, cwd=os.getcwd(), timeout=7200).returncode
print(f'[ours] infer rc={_rc} took {(time.time()-_t)/60:.1f} min', flush=True)
assert _rc == 0, f'ours infer rc={_rc}'

In [ ]:

# ==== Stage D: per-label rank fusion (atomic swap; fallback keeps the nartaa base) ====
import os, glob, shutil, time
import numpy as np, pandas as pd
from scipy.stats import rankdata

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]

# shrunk per-label weights from the 58-gold gate (cols: cnxt, nartaa, ours)
W_SHRUNK = np.array([
    [0.32, 0.52, 0.17], [0.32, 0.37, 0.32], [0.17, 0.62, 0.22],
    [0.42, 0.37, 0.22], [0.37, 0.32, 0.32], [0.42, 0.17, 0.42],
    [0.27, 0.57, 0.17], [0.17, 0.57, 0.27], [0.17, 0.67, 0.17],
    [0.42, 0.37, 0.22], [0.62, 0.17, 0.22], [0.22, 0.57, 0.22],
], dtype=np.float64)

def rank_pct(df):
    out = pd.DataFrame(index=df.index)
    for c in TARGETS:
        x = df[c].to_numpy(dtype=np.float64)
        r = rankdata(x) / (len(x) + 1e-9)
        out[c] = np.clip(r, 0.001, 0.999)
    return out

try:
    mode = os.environ.get('RSNA_STACK_MODE', 'shrunk')
    if mode == 'shrunk':
        W = W_SHRUNK.copy()
    elif mode == 'uniform3':
        W = np.full((12, 3), 1.0 / 3.0)
    elif mode == 'uniform2':           # best pair on the gate: cnxt + nartaa
        W = np.array([[0.5, 0.5, 0.0]] * 12)
    else:
        raise ValueError(f'unknown RSNA_STACK_MODE {mode!r}')
    W = W / W.sum(axis=1, keepdims=True)
    print('[stack] mode', mode, '| weights (cnxt,nartaa,ours) per label:', flush=True)
    for t, w in zip(TARGETS, W):
        print(f'  {t:<15} {np.round(w, 2)}', flush=True)

    sub = pd.read_csv('/kaggle/working/submission.csv', dtype={'StudyInstanceUID': str})
    ids = sub['StudyInstanceUID'].tolist()
    assert len(ids) > 0 and np.isfinite(sub[TARGETS].to_numpy()).all(), 'nartaa base bad'

    def arm(path, name):
        a = pd.read_csv(path, dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID')
        a = a.reindex(ids)
        assert a[TARGETS].notna().all().all(), f'{name}: study mismatch vs submission'
        assert np.isfinite(a[TARGETS].to_numpy()).all(), f'{name}: non-finite'
        return rank_pct(a[TARGETS]).to_numpy()

    nartaa_r = rank_pct(sub[TARGETS]).to_numpy()
    cnxt_r = arm('/kaggle/working/arm_cnxt_test.csv', 'cnxt')
    ours_r = arm('/kaggle/working/arm_ours_test.csv', 'ours')

    def spear(a, b):
        return [np.corrcoef(rankdata(a[:, j]), rankdata(b[:, j]))[0, 1]
                for j in range(12)]
    for na, a, nb, b in (('cnxt', cnxt_r, 'nartaa', nartaa_r),
                         ('cnxt', cnxt_r, 'ours', ours_r),
                         ('ours', ours_r, 'nartaa', nartaa_r)):
        rho = spear(a, b)
        print(f'[stack] rho {na} vs {nb}: mean {np.mean(rho):.3f}', flush=True)

    fused = cnxt_r * W[:, 0] + nartaa_r * W[:, 1] + ours_r * W[:, 2]
    assert np.isfinite(fused).all(), 'non-finite fused values'
    fused_r = rank_pct(pd.DataFrame(fused, columns=TARGETS)).to_numpy()
    out = pd.DataFrame(fused_r, columns=TARGETS)
    out.insert(0, 'StudyInstanceUID', ids)
    assert out['StudyInstanceUID'].tolist() == ids, 'row identity drift'
    tmp = '/kaggle/working/_submission_stack.tmp'
    out.to_csv(tmp, index=False)
    os.replace(tmp, '/kaggle/working/submission.csv')
    shutil.copy('/kaggle/working/submission.csv', '/kaggle/working/stack3_submission.csv')
    print(f'[stack] OK mode={mode}: submission.csv replaced | {len(out)} rows x {len(out.columns)} cols',
          flush=True)
except Exception as e:
    print(f'[stack] FALLBACK ({type(e).__name__}: {e}) - keeping nartaa base', flush=True)
    if not os.path.exists('/kaggle/working/submission.csv') and os.path.exists('/kaggle/working/_sota_0949.csv'):
        shutil.copy('/kaggle/working/_sota_0949.csv', '/kaggle/working/submission.csv')
print('[stack] done', flush=True)